# MailMate AI: 100% Complete Project Codebase

This notebook contains the **entire** source code for the MailMate project. It is structured into clear modules for Backend logic, AI integration, and Frontend assets.

---

## 🛠️ AI Logic & Function Calling (`function_call.py`)


In [ ]:
import google.generativeai as genai
import pandas as pd
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

API_KEY = os.getenv("API_KEY")
if API_KEY:
    genai.configure(api_key=API_KEY)

functions = [
    {
        "name" : "create_email_analysis",
        "description" : "analyze an email and decide if it needs a reminder. Return the reminder text, date, category, sentiment, urgency and check if it is spam.",
        "parameters" : {"type": "object", 
                        "properties": {
                            "reminder": {
                                "type": "string",
                                "description": "text of reminder"
                            },
                            "reminder_date" : {
                                "type" : "string",
                                "description" : "date of the reminder in MM-DD-YYYY format or else just give none"
                            },
                            "category": {
                                "type": "string",
                                "enum": [
                                    "Work", "Education", "Finance", "Promotions", "Personal", "Support", "Updates", "Spam", "Other"
                                ],
                                "description": "category or type of the email" # give examples in the string if not accurate
                            },
                            "sentiment": {
                                "type": "string",
                                "description": "emotional tone of the email" # give examples in the string if not accurate
                            },
                            "urgency": {
                                "type": "string",
                                "enum": [
                                    "high", "low", "moderate"
                                ],
                                "description": "urgency of the email"
                            },
                            "spam": {
                                "type": "string",
                                "enum": [
                                    "true", "false"
                                ],
                                "description": "true if email is spam, false if email is not spam"
                            }
                        },
                        "required" : ["reminder", "reminder_date", "category", "sentiment", "urgency", "spam"]
        }
    }
]

def extract_data(subject, body):
    model = genai.GenerativeModel(model_name="models/gemini-1.5-flash-8b", tools = [
        {
            "function_declarations": functions
        }
    ])
    prompt = f"""Analyze the email and call the create_email_analysis function with the extracted details.
    Subject: {subject}
    Body: {body}"""
    try:
        response = model.generate_content(prompt)
        if response.candidates:
            parts = response.candidates[0].content.parts
            for part in parts:
                if "function_call" in part:
                    fn_call = part.function_call
                    args = fn_call.args
                    return {
                            "spam": args["spam"] if "spam" in args else False,
                            "reminder": args["reminder"] if "reminder" in args else "",
                            "reminder_date": args["reminder_date"] if "reminder_date" in args else "",
                            "category": args["category"] if "category" in args else "Other",
                                    "sentiment": args["sentiment"] if "sentiment" in args else "Neutral",
                                    "urgency": args["urgency"] if "urgency" in args else "Low",
                                }    
    except Exception as e:
        print("error", e)
    return None

def run_function_call(df):
    cols=["spam", "reminder", "reminder_date", "category", "sentiment", "urgency"]
    for col in cols:
        if col not in df:
            df[col] = ""
        df[col] = df[col].astype(object)
    new_rows=df[df["spam"].isna()]
    for idx,row in new_rows.iterrows():
        result=extract_data(row["subject"], row["body"])
        if result:
            for key, value in result.items():
                df.at[idx,key]=value
    return df

## 🛠️ Gmail Core Fetcher (`fetch/gmail_fetch.py`)


In [ ]:
import os, base64
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
import email
import pandas as pd

# Gmail Fetch Core Logic

def saveToExcel(df, filepath):
    df.to_excel(filepath, index=False)
    print(f"Emails saved to {filepath}")

def list_messages(service, q=None, label_ids=None, max_results=10):
    response = service.users().messages().list(userId='me', q=q, labelIds=label_ids, maxResults=max_results).execute()
    return response.get('messages', [])

def get_message(service, msg_id):
    return service.users().messages().get(userId='me', id=msg_id, format='full').execute()

def get_payload_text(payload):
    # Recursive walk to find text/plain or base64 body
    if 'parts' in payload:
        for part in payload['parts']:
            text = get_payload_text(part)
            if text:
                return text
    else:
        mime_type = payload.get('mimeType', '')
        body = payload.get('body', {}).get('data')
        if body:
            data = base64.urlsafe_b64decode(body.encode('UTF-8'))
            if mime_type == 'text/plain' or mime_type.startswith('text/'):
                return data.decode('utf-8', errors='replace')
            else:
                # Return raw for other types
                return data.decode('utf-8', errors='replace')
    return None

def download_attachments(service, msg):
    parts = msg.get('payload', {}).get('parts', [])
    for p in parts:
        filename = p.get('filename')
        body = p.get('body', {})
        if filename:
            att_id = body.get('attachmentId')
            if att_id:
                try:
                    att = service.users().messages().attachments().get(
                        userId='me', messageId=msg['id'], id=att_id
                    ).execute()
                    data = base64.urlsafe_b64decode(att['data'].encode('UTF-8'))
                    with open(filename, 'wb') as f:
                        f.write(data)
                    print('Saved attachment', filename)
                except Exception as e:
                    print(f"Error downloading attachment {filename}: {e}")

def load_existingEmails(filepath):
    if os.path.exists(filepath):
        return pd.read_excel(filepath, engine="openpyxl")
    else:
        return pd.DataFrame(columns=["id", "date", "from", "subject", "body"])

def main(creds=None, filepath="email.xlsx"):
    if not creds:
        print("Error: No credentials provided to main")
        return
        
    service = build('gmail', 'v1', credentials=creds)
    msgs = list_messages(service, q='is:unread', label_ids=None)
    print(f'Found {len(msgs)} messages')
    new_emails = []
    existing_df = load_existingEmails(filepath)
    
    # Create a mapping of id to date for quick lookup
    id_to_date = {str(row['id']): str(row.get('date', '')) for _, row in existing_df.iterrows()}
    
    for m in msgs:
        msg_id = str(m["id"])
        if msg_id in id_to_date:
            existing_date = id_to_date[msg_id]
            if existing_date and existing_date not in ["", "nan", "No Date", "n/a"]:
                continue
        
        try:
            msg = get_message(service, m['id'])
            headers = {h['name']: h['value'] for h in msg['payload'].get('headers', [])}
            text = get_payload_text(msg['payload'])
            
            # download attachments if any
            download_attachments(service, msg)
            
            new_emails.append({
                "id" : m["id"],
                "date" : headers.get("Date", "n/a"),
                "from" : headers.get("From", "n/a"),
                "subject" : headers.get("Subject", "n/a"),
                "body" : text
            })
        except Exception as e:
            print(f"Error fetching message {m['id']}: {e}")

    if new_emails:
        df=pd.concat([existing_df, pd.DataFrame(new_emails)], ignore_index=True)
        # Keep the record with the most data (re-fetched date) if duplicates exist
        df.drop_duplicates(subset=['id'], keep='last', inplace=True)
        saveToExcel(df, filepath)


## 🛠️ Main Orchestrator (`scheduler.py`)


In [ ]:
import os
import time
import threading
from datetime import datetime, timedelta
from flask import Flask, render_template, jsonify, request, redirect, url_for, send_from_directory, session
import pandas as pd
from fetch.gmail_fetch import main as fetch_gmail
from function_call import run_function_call
from collections import Counter
import google.generativeai as genai
from functools import wraps
from dotenv import load_dotenv
from flask_mail import Mail, Message
from google_auth_oauthlib.flow import Flow
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
import json

os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = '1'

load_dotenv()

app = Flask(__name__, template_folder="templates")
app.secret_key = os.getenv('SECRET_KEY', 'your-secret-key-change-this-in-production')
base_dir = os.path.dirname(os.path.abspath(__file__))

app.config["MAIL_SERVER"]="smtp.gmail.com"
app.config["MAIL_PORT"]=587
app.config["MAIL_USE_TLS"]=True
app.config["MAIL_USE_SSL"]=False
app.config["MAIL_USERNAME"]="devansh.malhotra2027@gmail.com"
app.config["MAIL_PASSWORD"]="oupe afur cgeh xrio"
app.config["MAIL_DEFAULT_SENDER"]=("MailMate", "devansh.malhotra2027@gmail.com")
mail=Mail(app)

API_KEY = os.getenv("API_KEY")
if not API_KEY:
    raise ValueError("API_KEY not found in .env file!")
genai.configure(api_key=API_KEY)
DATA_DIR = 'user_data'
USERS_FILE = 'users.json'
USERS = {}

if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

SCOPES = ['https://www.googleapis.com/auth/gmail.readonly', 'https://www.googleapis.com/auth/gmail.send']

def load_users():
    global USERS
    if os.path.exists(USERS_FILE):
        import json
        with open(USERS_FILE, 'r') as f:
            USERS = json.load(f)

def save_users():
    import json
    with open(USERS_FILE, 'w') as f:
        json.dump(USERS, f, indent=4)
load_users()

def get_user_token_path(username):
    return os.path.join(DATA_DIR, f"{username}_token.json")

def get_user_data_path(username):
    return os.path.join(DATA_DIR, f"{username}_emails.xlsx")

def login_required(f):
    @wraps(f)
    def decorated_function(*args, **kwargs):
        if 'username' not in session:
            return redirect(url_for('login'))
        return f(*args, **kwargs)
    return decorated_function

pipeline_state = {
    'running': True,
    'last_update': None,
    'total_emails': 0,
    'processed_today': 0
}

@app.route('/login', methods=['GET', 'POST'])
def login():
    """Login page"""
    if request.method == 'POST':
        username = request.form.get('username')
        password = request.form.get('password')
        
        if username and password and username in USERS and USERS[username] == password:
            session['username'] = username
            return redirect(url_for('index'))
        else:
            return render_template('login.html', error='Invalid username or password')
    
    # If already logged in, redirect to index
    if 'username' in session:
        return redirect(url_for('index'))
    
    return render_template('login.html')

@app.route('/logout')
def logout():
    """Logout user"""
    session.pop('username', None)
    return redirect(url_for('login'))

@app.route('/signup', methods=['GET', 'POST'])
def signup():
    if request.method == 'POST':
        username = request.form.get('username')
        password = request.form.get('password')
        confirm_password = request.form.get('confirm_password')
        
        if not username or not password:
            return render_template('signup.html', error='Username and password are required')
        
        if len(username) < 3:
            return render_template('signup.html', error='Username must be at least 3 characters')
        
        if len(password) < 6:
            return render_template('signup.html', error='Password must be at least 6 characters')
        
        if password != confirm_password:
            return render_template('signup.html', error='Passwords do not match')
        
        if username in USERS:
            return render_template('signup.html', error='Username already exists')
        
        USERS[username] = password
        save_users()
        
        session['username'] = username
        return redirect(url_for('index'))
    
    if 'username' in session:
        return redirect(url_for('index'))
    
    return render_template('signup.html')

@app.route('/authorize')
@login_required
def authorize():
    flow = Flow.from_client_secrets_file(
        'credentials.json',
        scopes=SCOPES,
        redirect_uri=url_for('oauth2callback', _external=True)
    )
    authorization_url, state = flow.authorization_url(
        access_type='offline',
        include_granted_scopes='true'
    )
    session['oauth_state'] = state
    return redirect(authorization_url)

@app.route('/oauth2callback')
@login_required
def oauth2callback():
    state = session['oauth_state']
    flow = Flow.from_client_secrets_file(
        'credentials.json',
        scopes=SCOPES,
        state=state,
        redirect_uri=url_for('oauth2callback', _external=True)
    )
    authorization_response = request.url
    flow.fetch_token(authorization_response=authorization_response)
    
    credentials = flow.credentials
    token_path = get_user_token_path(session['username'])
    with open(token_path, 'w') as f:
        f.write(credentials.to_json())
    
    return redirect(url_for('index'))

@app.route('/style.css')
def serve_css():
    return send_from_directory(os.path.join(base_dir, 'templates'), 'style.css', mimetype='text/css')

def run_pipeline_for_user(username):
    try:
        token_path = get_user_token_path(username)
        if not os.path.exists(token_path):
            return
            
        print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Running pipeline for {username}...")
        
        creds = Credentials.from_authorized_user_file(token_path, SCOPES)
        if not creds.valid:
            if creds.expired and creds.refresh_token:
                from google.auth.transport.requests import Request
                creds.refresh(Request())
                with open(token_path, 'w') as f:
                    f.write(creds.to_json())
            else:
                print(f"Token invalid for {username}")
                return

        data_path = get_user_data_path(username)
        
        # Fetch emails from Gmail
        fetch_gmail(creds=creds, filepath=data_path)
        
        # Read and process emails
        if os.path.exists(data_path):
            df = pd.read_excel(data_path)
            df = run_function_call(df)
            
            df.to_excel(data_path, index=False)
            
            # Update state (global state per user would be better, but keeping it simple)
            pipeline_state['last_update'] = datetime.now()
            pipeline_state['total_emails'] = len(df)
            pipeline_state['processed_today'] = len(df[df['spam'].notna()])
        
        print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Pipeline completed for {username}.")
        
    except Exception as e:
        print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Pipeline error for {username}: {e}")

def run_pipeline():
    for username in USERS:
        run_pipeline_for_user(username)

def pipeline_loop():
    while True:
        if pipeline_state['running']:
            run_pipeline()
        time.sleep(600)  # 10 minutes

@app.route('/')
@login_required
def index():
    try:
        data_path = get_user_data_path(session['username'])
        if os.path.exists(data_path):
            df = pd.read_excel(data_path)
            emails = df.to_dict('records')
        else:
            emails = []
        
        # Check if user has connected Gmail
        has_gmail = os.path.exists(get_user_token_path(session['username']))
        
        return render_template('index.html', emails=emails, has_gmail=has_gmail)
    
    except Exception as e:
        print(f"Error loading emails: {e}")
        return render_template('index.html', emails=[])

@app.route('/analysis')
@login_required
def analysis():
    try:
        data_path = get_user_data_path(session['username'])
        if os.path.exists(data_path):
            df = pd.read_excel(data_path)
            
            # Calculate sentiment counts
            positive_count = len(df[df['sentiment'].astype(str).str.lower().str.contains('positive', na=False)])
            negative_count = len(df[df['sentiment'].astype(str).str.lower().str.contains('negative', na=False)])
            neutral_count = len(df) - positive_count - negative_count
            
            # Get category distribution
            categories = Counter(df['category'].dropna())
            category_labels = list(categories.keys())
            category_data = list(categories.values())
            
            # Get urgency distribution
            urgency_counter = Counter(df['urgency'].dropna())
            urgency_labels = list(urgency_counter.keys())
            urgency_data = list(urgency_counter.values())
            
            # Timeline data (emails per day for last 7 days)
            timeline_labels = [(datetime.now() - timedelta(days=i)).strftime('%b %d') for i in range(6, -1, -1)]
            timeline_data = [5, 8, 6, 9, 7, 10, 12]
            
            return render_template('analysis.html',
                                 positive_count=positive_count,
                                 negative_count=negative_count,
                                 neutral_count=neutral_count,
                                 categories=categories,
                                 category_labels=category_labels,
                                 category_data=category_data,
                                 urgency_labels=urgency_labels,
                                 urgency_data=urgency_data,
                                 timeline_labels=timeline_labels,
                                 timeline_data=timeline_data)
        else:
            return render_template('analysis.html',
                                 positive_count=0,
                                 negative_count=0,
                                 neutral_count=0,
                                 categories={},
                                 category_labels=[],
                                 category_data=[],
                                 urgency_labels=[],
                                 urgency_data=[],
                                 timeline_labels=[],
                                 timeline_data=[])
    
    except Exception as e:
        print(f"Error loading analysis: {e}")
        return render_template('analysis.html',
                             positive_count=0,
                             negative_count=0,
                             neutral_count=0,
                             categories={},
                             category_labels=[],
                             category_data=[],
                             urgency_labels=[],
                             urgency_data=[],
                             timeline_labels=[],
                             timeline_data=[])

@app.route('/reminders')
@login_required
def reminders():
    try:
        data_path = get_user_data_path(session['username'])
        if os.path.exists(data_path):
            df = pd.read_excel(data_path)
            
            # Filter emails that have reminders
            reminder_df = df[df['reminder'].notna() & (df['reminder'] != '')]
            reminders_list = reminder_df.to_dict('records')
            
            # Calculate stats
            urgent_reminders = len(reminder_df[reminder_df.get('urgency', '') == 'high'])
            
            # Count upcoming reminders (this week)
            upcoming_reminders = len(reminder_df)
            
            return render_template('reminders.html',
                                 reminders=reminders_list,
                                 urgent_reminders=urgent_reminders,
                                 upcoming_reminders=upcoming_reminders)
        else:
            return render_template('reminders.html',
                                 reminders=[],
                                 urgent_reminders=0,
                                 upcoming_reminders=0)
    
    except Exception as e:
        print(f"Error loading reminders: {e}")
        return render_template('reminders.html',
                             reminders=[],
                             urgent_reminders=0,
                             upcoming_reminders=0)

@app.route('/spam')
@login_required
def spam():
    try:
        data_path = get_user_data_path(session['username'])
        if os.path.exists(data_path):
            df = pd.read_excel(data_path)
            
            # Filter spam emails
            spam_df = df[df['spam'].astype(str).str.lower() == 'true']
            spam_list = spam_df.to_dict('records')
            
            # Calculate stats
            total_emails = len(df)
            spam_count = len(spam_df)
            safe_emails = total_emails - spam_count
            spam_percentage = round((spam_count / total_emails * 100) if total_emails > 0 else 0, 1)
            
            return render_template('spam.html',
                                 spam_emails=spam_list,
                                 safe_emails=safe_emails,
                                 spam_percentage=spam_percentage)
        else:
            return render_template('spam.html',
                                 spam_emails=[],
                                 safe_emails=0,
                                 spam_percentage=0)
    
    except Exception as e:
        print(f"Error loading spam: {e}")
        return render_template('spam.html',
                             spam_emails=[],
                             safe_emails=0,
                             spam_percentage=0)

@app.route('/api/refresh', methods=['POST'])
def refresh_emails():
    try:
        run_pipeline()
        return redirect(request.referrer or url_for('index'))
    except Exception as e:
        print(f"Error: {e}")
        return redirect(request.referrer or url_for('index'))

@app.route('/api/stats')
def get_stats():
    try:
        data_path = get_user_data_path(session.get('username'))
        if os.path.exists(data_path):
            df = pd.read_excel(data_path)
            total = len(df)
            processed = len(df[df['spam'].notna()])
        else:
            total = 0
            processed = 0
        
        return jsonify({
            'total_emails': total,
            'processed_today': processed,
            'last_update': pipeline_state['last_update'].strftime('%H:%M:%S') if pipeline_state['last_update'] else '--:--',
            'pipeline_running': pipeline_state['running']
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/api/emails')
def get_emails():
    try:
        data_path = get_user_data_path(session.get('username'))
        if os.path.exists(data_path):
            df = pd.read_excel(data_path)
            emails = df.to_dict('records')
            return jsonify({'emails': emails})
        else:
            return jsonify({'emails': []})
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/api/send_email', methods=['POST', 'GET'])
def send_email():
    try:
        data = request.json
        if not data:
            return jsonify({'success': False, 'message': 'Invalid request data'}), 400
        recipient = data.get('recipient')
        subject = data.get('subject')
        body = data.get('body')
        print(recipient)
        msg=Message(
            subject=subject,
            recipients=[recipient],
            body=body
        )

        mail.send(msg)

        # Log to file
        log_file = "sent_emails.xlsx"
        
        email_data = {
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'recipient': recipient,
            'subject': subject,
            'body': body,
            'status': 'logged'
        }
        
        if os.path.exists(log_file):
            df = pd.read_excel(log_file, engine="openpyxl")
            df = pd.concat([df, pd.DataFrame([email_data])], ignore_index=True)
        else:
            df = pd.DataFrame([email_data])
        
        df.to_excel(log_file, index=False)
        
        print(f"[EMAIL SENT] To: {recipient}, Subject: {subject}")
        
        return jsonify({'success': True, 'message': 'Email logged successfully'})
    
    except Exception as e:
        print(f"Error logging email: {e}")
        return jsonify({'success': False, 'message': str(e)}), 500

@app.route('/api/summarize', methods=['POST'])
@login_required
def summarize_email():
    try:
        data = request.json
        if not data:
            return jsonify({'success': False, 'message': 'Invalid request data'}), 400
        body = data.get('body')
        
        prompt = f"""Summarize this email concisely in 2-3 bullet points:
        
{body}"""
        
        model = genai.GenerativeModel("models/gemma-3-12b-it")
        response = model.generate_content(prompt)
        
        return jsonify({
            'success': True,
            'summary': response.text
        })
    except Exception as e:
        print(f"Error summarizing: {e}")
        return jsonify({'success': False, 'message': str(e)}), 500

@app.route('/api/clean_text', methods=['POST'])
@login_required
def clean_text():
    try:
        data = request.json
        if not data:
            return jsonify({'success': False, 'message': 'Invalid request data'}), 400
        body = data.get('body')
        
        prompt = f"""Extract only the main natural language text from this email suitable for reading aloud. 
        Remove all URLs, 'View image', 'Follow link', 'Caption', repetitive dashes/dividers, header/footer navigation, and technical metadata.
        Format it as clean, readable paragraphs.
        
        Email Content:
        {body}"""
        
        model = genai.GenerativeModel("models/gemma-3-12b-it")
        response = model.generate_content(prompt)
        
        return jsonify({
            'success': True,
            'cleaned_text': response.text
        })
    except Exception as e:
        print(f"Error cleaning text: {e}")
        return jsonify({'success': False, 'message': str(e)}), 500

@app.route('/api/generate_ai_email', methods=['POST'])
def generate_ai_email():
    try:
        data = request.json
        if not data:
            return jsonify({'success': False, 'message': 'Invalid request data'}), 400
        email_type = data.get('email_type')
        purpose = data.get('purpose')
        
        # Create prompt for Gemini
        prompt = f"""Generate a {email_type} email based on the following purpose:

{purpose}

Please write a professional, well-structured email. Include appropriate greeting, body, and closing."""
        
        model = genai.GenerativeModel("models/gemma-3-12b-it")
        response = model.generate_content(prompt)
        
        generated_email = response.text
        
        # Log the generation
        log_file = "ai_generated_emails.xlsx"
        
        log_data = {
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'email_type': email_type,
            'purpose': purpose,
            'generated_email': generated_email
        }
        
        if os.path.exists(log_file):
            df = pd.read_excel(log_file)
            df = pd.concat([df, pd.DataFrame([log_data])], ignore_index=True)
        else:
            df = pd.DataFrame([log_data])
        
        df.to_excel(log_file, index=False)
        
        print(f"[AI EMAIL GENERATED] Type: {email_type}")
        
        return jsonify({
            'success': True,
            'email': generated_email
        })
    
    except Exception as e:
        print(f"Error generating AI email: {e}")
        return jsonify({'success': False, 'message': str(e)}), 500

@app.route('/api/generate_todo', methods=['POST'])
def generate_todo():
    try:
        data_path = get_user_data_path(session.get('username'))
        if not os.path.exists(data_path):
            return jsonify({'success': False, 'message': 'No emails found'}), 404
        
        df = pd.read_excel(data_path)
        
        # Filter reminders
        reminder_df = df[df['reminder'].notna() & (df['reminder'] != '')]
        
        # Get today's date
        today = datetime.now().strftime('%Y-%m-%d')
        
        # Filter today's reminders (if date field exists)
        today_reminders = []
        for _, row in reminder_df.iterrows():
            date_str = str(row.get('date', ''))
            if today in date_str or date_str == 'none' or pd.isna(row.get('date')):
                today_reminders.append({
                    'reminder': row['reminder'],
                    'urgency': row.get('urgency', 'low'),
                    'category': row.get('category', 'Other'),
                    'from': row.get('from', 'Unknown')
                })
        
        if not today_reminders:
            return jsonify({'success': False, 'message': 'No reminders for today'}), 404
        
        # Create prompt for AI enhancement
        reminders_text = "\n".join([f"- {r['reminder']} (Urgency: {r['urgency']}, Category: {r['category']})" 
                                    for r in today_reminders])
        
        prompt = f"""Based on these email reminders, create a prioritized to-do list:

{reminders_text}

Please organize them by priority, add estimated time for each task, and suggest the best order to complete them. Format as a clear, actionable to-do list."""
        
        model = genai.GenerativeModel("models/gemma-3-12b-it")
        response = model.generate_content(prompt)
        
        ai_todo_list = response.text
        
        # Create Excel file with to-do list
        filename = f"todo_list_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
        
        todo_data = []
        for i, reminder in enumerate(today_reminders, 1):
            todo_data.append({
                'Task #': i,
                'Task': reminder['reminder'],
                'Urgency': reminder['urgency'],
                'Category': reminder['category'],
                'Source': reminder['from'],
                'Status': 'Pending'
            })
        
        # Add AI suggestions as a separate sheet
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            pd.DataFrame(todo_data).to_excel(writer, sheet_name='Tasks', index=False)
            pd.DataFrame([{'AI Suggestions': ai_todo_list}]).to_excel(writer, sheet_name='AI Suggestions', index=False)
        
        print(f"[TO-DO LIST GENERATED] File: {filename}, Tasks: {len(today_reminders)}")
        
        return jsonify({
            'success': True,
            'filename': filename,
            'task_count': len(today_reminders),
            'tasks': todo_data,
            'ai_todo': ai_todo_list
        })
    
    except Exception as e:
        print(f"Error generating to-do list: {e}")
        return jsonify({'success': False, 'message': str(e)}), 500

@app.route('/api/mark_complete', methods=['POST'])
def mark_complete():
    try:
        data = request.json
        if not data:
            return jsonify({'success': False, 'message': 'Invalid request data'}), 400
        email_id = data.get('email_id')
        
        data_path = get_user_data_path(session.get('username'))
        if not os.path.exists(data_path):
            return jsonify({'success': False, 'message': 'Email file not found'}), 404
        
        df = pd.read_excel(data_path)
        
        # Add completed column if it doesn't exist
        if 'completed' not in df.columns:
            df['completed'] = False
        
        # Mark as complete
        df.loc[df['id'] == email_id, 'completed'] = True
        df.loc[df['id'] == email_id, 'completed_date'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        df.to_excel(data_path, index=False)
        
        print(f"[REMINDER COMPLETED] ID: {email_id}")
        
        return jsonify({'success': True, 'message': 'Reminder marked as complete'})
    
    except Exception as e:
        print(f"Error marking complete: {e}")
        return jsonify({'success': False, 'message': str(e)}), 500

@app.route('/download/<filename>')
def download_file(filename):
    try:
        return send_from_directory('.', filename, as_attachment=True)
    except Exception as e:
        print(f"Error downloading file: {e}")
        return "File not found", 404

def main():
    pipeline_thread = threading.Thread(target=pipeline_loop, daemon=True)
    pipeline_thread.start()
    
    print("=" * 60)
    print("Email Pipeline Dashboard Starting")
    print("=" * 60)
    print(f"Dashboard URL: http://localhost:5000")
    print(f"Routes available:")
    print(f"  - / (Inbox)")
    print(f"  - /analysis (Email Analysis)")
    print(f"  - /reminders (Reminders & To-Do)")
    print(f"  - /spam (Spam Filter)")
    print(f"API Endpoints:")
    print(f"  - /api/send_email (Log sent emails)")
    print(f"  - /api/generate_ai_email (Generate AI emails)")
    print(f"  - /api/generate_todo (Generate to-do list)")
    print(f"Pipeline interval: 10 minutes")
    print("=" * 60)
    
    app.run(debug=True, host='0.0.0.0', port=5000, use_reloader=False)

if __name__ == "__main__":
    main()

## 🛠️ Design CSS (`templates/style.css`)


In [ ]:
* {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
        }

        .container {
            display: flex;
            min-height: 100vh;
        }

        /* Sidebar */
        .sidebar {
            width: 260px;
            background: rgba(255, 255, 255, 0.95);
            backdrop-filter: blur(10px);
            padding: 20px;
            display: flex;
            flex-direction: column;
            box-shadow: 2px 0 10px rgba(0, 0, 0, 0.1);
        }

        .logo {
            display: flex;
            align-items: center;
            gap: 12px;
            margin-bottom: 40px;
            padding-bottom: 20px;
            border-bottom: 2px solid #e0e0e0;
        }

        .logo i {
            font-size: 32px;
            color: #667eea;
        }

        .logo h2 {
            font-size: 22px;
            color: #2d3748;
        }

        .nav-menu {
            display: flex;
            flex-direction: column;
            gap: 8px;
            flex: 1;
        }

        .nav-item {
            display: flex;
            align-items: center;
            gap: 15px;
            padding: 14px 18px;
            border-radius: 12px;
            text-decoration: none;
            color: #4a5568;
            font-weight: 500;
            transition: all 0.3s ease;
        }

        .nav-item:hover {
            background: #f7fafc;
            color: #667eea;
            transform: translateX(5px);
        }

        .nav-item.active {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
        }

        .nav-item i {
            font-size: 18px;
            width: 20px;
        }

        .sidebar-footer {
            margin-top: auto;
            padding-top: 20px;
            border-top: 2px solid #e0e0e0;
        }

        .refresh-btn {
            width: 100%;
            padding: 12px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            border: none;
            border-radius: 10px;
            font-size: 14px;
            font-weight: 600;
            cursor: pointer;
            display: flex;
            align-items: center;
            justify-content: center;
            gap: 10px;
            transition: all 0.3s ease;
            margin-bottom: 12px;
        }

        .refresh-btn:hover {
            transform: translateY(-2px);
            box-shadow: 0 4px 12px rgba(102, 126, 234, 0.4);
        }

        .refresh-btn.spinning i {
            animation: spin 1s linear infinite;
        }

        @keyframes spin {
            from { transform: rotate(0deg); }
            to { transform: rotate(360deg); }
        }

        .last-update {
            font-size: 12px;
            color: #718096;
            text-align: center;
        }

        /* Main Content */
        .main-content {
            flex: 1;
            padding: 30px;
            overflow-y: auto;
        }

        .header {
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 30px;
        }

        .header-left h1 {
            color: white;
            font-size: 32px;
            margin-bottom: 5px;
        }

        .subtitle {
            color: rgba(255, 255, 255, 0.8);
            font-size: 14px;
        }

        .search-box {
            position: relative;
            width: 350px;
        }

        .search-box i {
            position: absolute;
            left: 15px;
            top: 50%;
            transform: translateY(-50%);
            color: #a0aec0;
        }

        .search-box input {
            width: 100%;
            padding: 12px 15px 12px 45px;
            border: none;
            border-radius: 12px;
            background: rgba(255, 255, 255, 0.95);
            font-size: 14px;
            outline: none;
            transition: all 0.3s ease;
        }

        .search-box input:focus {
            box-shadow: 0 4px 12px rgba(0, 0, 0, 0.15);
        }

        /* Stats Grid */
        .stats-grid {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(250px, 1fr));
            gap: 20px;
            margin-bottom: 30px;
        }

        .stat-card {
            background: rgba(255, 255, 255, 0.95);
            border-radius: 16px;
            padding: 24px;
            display: flex;
            align-items: center;
            gap: 20px;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
            transition: all 0.3s ease;
        }

        .stat-card:hover {
            transform: translateY(-5px);
            box-shadow: 0 8px 16px rgba(0, 0, 0, 0.15);
        }

        .stat-icon {
            width: 60px;
            height: 60px;
            border-radius: 14px;
            display: flex;
            align-items: center;
            justify-content: center;
            font-size: 24px;
            color: white;
        }

        .stat-icon.blue {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        }

        .stat-icon.green {
            background: linear-gradient(135deg, #48bb78 0%, #38a169 100%);
        }

        .stat-icon.orange {
            background: linear-gradient(135deg, #ed8936 0%, #dd6b20 100%);
        }

        .stat-icon.red {
            background: linear-gradient(135deg, #f56565 0%, #e53e3e 100%);
        }

        .stat-info h3 {
            font-size: 32px;
            color: #2d3748;
            margin-bottom: 5px;
        }

        .stat-info p {
            color: #718096;
            font-size: 14px;
            font-weight: 500;
        }

        /* Filter Tabs */
        .filter-tabs {
            display: flex;
            gap: 10px;
            margin-bottom: 25px;
            flex-wrap: wrap;
        }

        .tab-btn {
            padding: 10px 20px;
            background: rgba(255, 255, 255, 0.9);
            border: 2px solid transparent;
            border-radius: 10px;
            color: #4a5568;
            font-size: 14px;
            font-weight: 600;
            cursor: pointer;
            transition: all 0.3s ease;
        }

        .tab-btn:hover {
            background: white;
            border-color: #667eea;
        }

        .tab-btn.active {
            background: white;
            border-color: #667eea;
            color: #667eea;
        }

        /* Email List */
        .email-list {
            display: grid;
            gap: 20px;
        }

        .email-card {
            background: rgba(255, 255, 255, 0.95);
            border-radius: 16px;
            padding: 20px;
            cursor: pointer;
            transition: all 0.3s ease;
            box-shadow: 0 2px 8px rgba(0, 0, 0, 0.1);
        }

        .email-card:hover {
            transform: translateY(-3px);
            box-shadow: 0 8px 20px rgba(0, 0, 0, 0.15);
        }

        .email-header {
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 15px;
        }

        .email-sender {
            display: flex;
            gap: 15px;
            align-items: center;
        }

        .avatar {
            width: 50px;
            height: 50px;
            border-radius: 50%;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            display: flex;
            align-items: center;
            justify-content: center;
            color: white;
            font-size: 20px;
            font-weight: bold;
        }

        .sender-info h4 {
            font-size: 16px;
            color: #2d3748;
            margin-bottom: 3px;
        }

        .email-time {
            font-size: 12px;
            color: #a0aec0;
        }

        .email-badges {
            display: flex;
            gap: 8px;
            flex-wrap: wrap;
        }

        .badge {
            padding: 6px 12px;
            border-radius: 20px;
            font-size: 11px;
            font-weight: 600;
            display: inline-flex;
            align-items: center;
            gap: 5px;
        }

        .badge-danger {
            background: #fee;
            color: #e53e3e;
        }

        .badge-warning {
            background: #fef5e7;
            color: #dd6b20;
        }

        .badge-info {
            background: #e6f7ff;
            color: #2b6cb0;
        }

        .badge-category {
            background: #f0f4f8;
            color: #4a5568;
        }

        .email-content {
            margin-bottom: 15px;
        }

        .email-subject {
            font-size: 18px;
            color: #2d3748;
            margin-bottom: 8px;
            font-weight: 600;
        }

        .email-preview {
            color: #718096;
            font-size: 14px;
            line-height: 1.6;
        }

        .email-footer {
            display: flex;
            gap: 15px;
            align-items: center;
            padding-top: 12px;
            border-top: 1px solid #e2e8f0;
        }

        .sentiment-indicator {
            display: flex;
            align-items: center;
            gap: 6px;
            font-size: 13px;
            font-weight: 600;
            padding: 5px 12px;
            border-radius: 20px;
        }

        .sentiment-positive {
            background: #e6ffed;
            color: #38a169;
        }

        .sentiment-negative {
            background: #fee;
            color: #e53e3e;
        }

        .sentiment-neutral {
            background: #f0f4f8;
            color: #718096;
        }

        .reminder-indicator {
            display: flex;
            align-items: center;
            gap: 6px;
            font-size: 13px;
            color: #ed8936;
            font-weight: 600;
        }

        .charts-container {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(400px, 1fr));
            gap: 20px;
            margin-bottom: 30px;
        }

        .chart-card {
            background: rgba(255, 255, 255, 0.95);
            border-radius: 16px;
            padding: 25px;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
            height: 350px;
        }

        .chart-card h3 {
            color: #2d3748;
            margin-bottom: 20px;
            font-size: 18px;
        }

        .chart-card canvas {
            max-height: 280px;
        }

        .category-breakdown {
            background: rgba(255, 255, 255, 0.95);
            border-radius: 16px;
            padding: 25px;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
        }

        .category-breakdown h2 {
            color: #2d3748;
            margin-bottom: 20px;
            font-size: 22px;
        }

        .category-list {
            display: grid;
            gap: 15px;
        }

        .category-item {
            display: flex;
            justify-content: space-between;
            align-items: center;
            padding: 15px 20px;
            background: #f7fafc;
            border-radius: 10px;
            transition: all 0.3s ease;
        }

        .category-item:hover {
            background: #e6f7ff;
            transform: translateX(5px);
        }

        .category-name {
            display: flex;
            align-items: center;
            gap: 12px;
            color: #2d3748;
            font-weight: 600;
        }

        .category-name i {
            color: #667eea;
        }

        .category-count {
            color: #718096;
            font-size: 14px;
        }

        .spam-container {
            margin-top: 20px;
        }

        .section-header {
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 20px;
            flex-wrap: wrap;
            gap: 15px;
        }

        .section-title {
            color: white;
            font-size: 24px;
        }

        .filter-buttons {
            display: flex;
            gap: 10px;
        }

        .filter-btn {
            padding: 8px 16px;
            background: rgba(255, 255, 255, 0.9);
            border: 2px solid transparent;
            border-radius: 8px;
            color: #4a5568;
            font-size: 14px;
            font-weight: 600;
            cursor: pointer;
            transition: all 0.3s ease;
        }

        .filter-btn:hover,
        .filter-btn.active {
            background: white;
            border-color: #f56565;
            color: #f56565;
        }

        .spam-list {
            display: grid;
            gap: 20px;
        }

        .spam-card {
            background: rgba(255, 255, 255, 0.95);
            border-radius: 16px;
            padding: 20px;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
            border-left: 5px solid #f56565;
            transition: all 0.3s ease;
        }

        .spam-card:hover {
            transform: translateY(-3px);
            box-shadow: 0 8px 16px rgba(245, 101, 101, 0.2);
        }

        .spam-warning {
            background: linear-gradient(135deg, #f56565 0%, #e53e3e 100%);
            color: white;
            padding: 10px 15px;
            border-radius: 8px;
            font-size: 13px;
            font-weight: 700;
            display: inline-flex;
            align-items: center;
            gap: 8px;
            margin-bottom: 15px;
        }

        .spam-header {
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 15px;
        }

        .spam-sender {
            display: flex;
            gap: 15px;
            align-items: center;
        }

        .spam-badges {
            display: flex;
            gap: 8px;
            flex-wrap: wrap;
        }

        .spam-content {
            margin-bottom: 15px;
        }

        .spam-footer {
            display: flex;
            gap: 10px;
            padding-top: 15px;
            border-top: 1px solid #e2e8f0;
            flex-wrap: wrap;
        }

        .btn-danger {
            background: linear-gradient(135deg, #f56565 0%, #e53e3e 100%);
            color: white;
            border: none;
            padding: 10px 16px;
            border-radius: 8px;
            font-size: 13px;
            font-weight: 600;
            cursor: pointer;
            display: flex;
            align-items: center;
            gap: 6px;
            transition: all 0.3s ease;
        }

        .btn-danger:hover {
            transform: translateY(-2px);
            box-shadow: 0 4px 12px rgba(245, 101, 101, 0.4);
        }

        .btn-secondary {
            background: #718096;
            color: white;
            border: none;
            padding: 10px 16px;
            border-radius: 8px;
            font-size: 13px;
            font-weight: 600;
            cursor: pointer;
            display: flex;
            align-items: center;
            gap: 6px;
            transition: all 0.3s ease;
        }

        .btn-secondary:hover {
            background: #4a5568;
            transform: translateY(-2px);
        }

        .btn-view-spam {
            background: transparent;
            color: #667eea;
            border: 2px solid #667eea;
            padding: 10px 16px;
            border-radius: 8px;
            font-size: 13px;
            font-weight: 600;
            cursor: pointer;
            display: flex;
            align-items: center;
            gap: 6px;
            transition: all 0.3s ease;
            margin-left: auto;
        }

        .btn-view-spam:hover {
            background: #667eea;
            color: white;
        }

        .protection-info {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(280px, 1fr));
            gap: 20px;
            margin-top: 30px;
        }

        .info-card {
            background: rgba(255, 255, 255, 0.95);
            border-radius: 16px;
            padding: 25px;
            text-align: center;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
        }

        .info-card i {
            font-size: 40px;
            color: #667eea;
            margin-bottom: 15px;
        }

        .info-card h3 {
            color: #2d3748;
            font-size: 18px;
            margin-bottom: 10px;
        }

        .info-card p {
            color: #718096;
            font-size: 14px;
            line-height: 1.6;
        }

        .empty-state {
            text-align: center;
            padding: 80px 20px;
            background: rgba(255, 255, 255, 0.95);
            border-radius: 16px;
        }

        .empty-state i {
            font-size: 80px;
            color: #48bb78;
            margin-bottom: 20px;
        }

        .empty-state h3 {
            color: #2d3748;
            font-size: 24px;
            margin-bottom: 10px;
        }

        .empty-state p {
            color: #718096;
            font-size: 16px;
        }

        /* Modal */
        .modal {
            display: none;
            position: fixed;
            z-index: 1000;
            left: 0;
            top: 0;
            width: 100%;
            height: 100%;
            background: rgba(0, 0, 0, 0.6);
            backdrop-filter: blur(5px);
        }

        .modal-content {
            background: white;
            margin: 50px auto;
            padding: 0;
            border-radius: 20px;
            width: 90%;
            max-width: 800px;
            box-shadow: 0 20px 60px rgba(0, 0, 0, 0.3);
            animation: slideDown 0.3s ease;
        }

        @keyframes slideDown {
            from {
                transform: translateY(-50px);
                opacity: 0;
            }
            to {
                transform: translateY(0);
                opacity: 1;
            }
        }

        .modal-header {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 25px 30px;
            border-radius: 20px 20px 0 0;
            display: flex;
            justify-content: space-between;
            align-items: center;
        }

        .modal-header h2 {
            font-size: 22px;
            margin: 0;
        }

        .close {
            color: white;
            font-size: 32px;
            font-weight: bold;
            cursor: pointer;
            transition: all 0.3s ease;
            line-height: 1;
        }

        .close:hover {
            transform: rotate(90deg);
        }

        .modal-body {
            padding: 30px;
            max-height: 600px;
            overflow-y: auto;
        }

        .warning-box {
            background: #fee;
            border-left: 4px solid #f56565;
            padding: 15px;
            border-radius: 8px;
            margin-bottom: 20px;
            display: flex;
            align-items: center;
            gap: 10px;
            color: #742a2a;
        }

        .warning-box i {
            font-size: 20px;
            color: #f56565;
        }

        .detail-row {
            display: flex;
            gap: 15px;
            margin-bottom: 20px;
            align-items: flex-start;
        }

        .detail-row.full-width {
            flex-direction: column;
        }

        .detail-row strong {
            min-width: 120px;
            color: #4a5568;
            font-weight: 600;
        }

        .detail-row span {
            color: #2d3748;
        }

        .reminder-box {
            background: #fef5e7;
            border-left: 4px solid #ed8936;
            padding: 15px;
            border-radius: 8px;
            width: 100%;
        }

        .reminder-box p {
            color: #2d3748;
            margin-bottom: 8px;
            font-weight: 500;
        }

        .reminder-box span {
            font-size: 12px;
            color: #718096;
        }

        .email-body-full {
            background: #f7fafc;
            padding: 20px;
            border-radius: 10px;
            color: #2d3748;
            line-height: 1.8;
            white-space: pre-wrap;
            max-height: 400px;
            overflow-y: auto;
        }

        /* Responsive Design */
        @media (max-width: 768px) {
            .container {
                flex-direction: column;
            }
            
            .sidebar {
                width: 100%;
                padding: 15px;
            }
            
            .header {
                flex-direction: column;
                align-items: flex-start;
                gap: 15px;
            }
            
            .search-box {
                width: 100%;
            }
            
            .stats-grid {
                grid-template-columns: 1fr;
            }

            .charts-container {
                grid-template-columns: 1fr;
            }
        }

## 🛠️ Dashboard Template (`templates/index.html`)


In [ ]:
<!DOCTYPE html>
<html lang="en">

<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Email Manager Dashboard</title>
    <link rel="stylesheet" href="/style.css">
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
</head>

<body>
    <div class="container">
        <!-- Sidebar Navigation -->
        <aside class="sidebar">
            <div class="logo">
                <i class="fas fa-envelope-open-text"></i>
                <h2>Email Manager</h2>
            </div>
            <nav class="nav-menu">
                <a href="/" class="nav-item active" data-page="inbox">
                    <i class="fas fa-inbox"></i>
                    <span>Inbox</span>
                </a>
                <a href="/analysis" class="nav-item" data-page="analysis">
                    <i class="fas fa-chart-line"></i>
                    <span>Analysis</span>
                </a>
                <a href="/reminders" class="nav-item" data-page="reminders">
                    <i class="fas fa-bell"></i>
                    <span>Reminders</span>
                </a>
                <a href="/spam" class="nav-item" data-page="spam">
                    <i class="fas fa-shield-alt"></i>
                    <span>Spam</span>
                </a>
            </nav>
            <div class="sidebar-footer">
                <button class="refresh-btn" onclick="generateTodoList()"
                    style="background: linear-gradient(135deg, #48bb78 0%, #38a169 100%); margin-bottom: 10px;">
                    <i class="fas fa-magic"></i>
                    <span>Generate To-Do List</span>
                </button>
                <button class="refresh-btn" onclick="openSendEmailModal()">
                    <i class="fas fa-paper-plane"></i>
                    <span>Send Email</span>
                </button>
                <button class="refresh-btn" onclick="openAIEmailModal()"
                    style="background: linear-gradient(135deg, #9f7aea 0%, #805ad5 100%); margin-top: 10px;">
                    <i class="fas fa-robot"></i>
                    <span>Generate AI Email</span>
                </button>
                <button class="refresh-btn" onclick="refreshEmails()" style="margin-top: 10px;">
                    <i class="fas fa-sync-alt"></i>
                    <span>Refresh Emails</span>
                </button>
                <button class="refresh-btn" onclick="window.location.href='/logout'"
                    style="background: linear-gradient(135deg, #f56565 0%, #e53e3e 100%); margin-top: 10px;">
                    <i class="fas fa-sign-out-alt"></i>
                    <span>Logout</span>
                </button>
                <p class="last-update">Last update: <span id="lastUpdate">--:--</span></p>
            </div>
        </aside>

        <!-- Main Content -->
        <main class="main-content">
            <!-- Header -->
            <header class="header">
                <div class="header-left">
                    <h1 id="pageTitle">Inbox</h1>
                    <p class="subtitle">Manage your emails efficiently</p>
                </div>
                <div class="header-right" style="display: flex; align-items: center; gap: 20px;">
                    {% if not has_gmail %}
                    <div class="connection-banner"
                        style="background: rgba(245, 101, 101, 0.1); padding: 10px 20px; border-radius: 10px; border: 1px solid #f56565; display: flex; align-items: center; gap: 15px;">
                        <span style="color: #c53030; font-weight: 600;"><i class="fas fa-exclamation-triangle"></i>
                            Gmail not connected</span>
                        <a href="/authorize" class="header-btn"
                            style="background: #f56565; color: white; padding: 8px 15px; border-radius: 8px; text-decoration: none; font-size: 14px;">Connect
                            Gmail</a>
                    </div>
                    {% else %}
                    <div class="connection-status"
                        style="background: rgba(72, 187, 120, 0.1); padding: 5px 15px; border-radius: 20px; border: 1px solid #48bb78; color: #2f855a; font-size: 13px; font-weight: 600;">
                        <i class="fas fa-check-circle"></i> Gmail Connected
                    </div>
                    {% endif %}
                    <div class="header-right">
                        <div class="search-box">
                            <i class="fas fa-search"></i>
                            <input type="text" id="searchInput" placeholder="Search emails..." onkeyup="filterEmails()">
                        </div>
                    </div>
            </header>

            <!-- Stats Cards -->
            <div class="stats-grid">
                <div class="stat-card">
                    <div class="stat-icon blue">
                        <i class="fas fa-envelope"></i>
                    </div>
                    <div class="stat-info">
                        <h3 id="totalEmails">{{ emails|length }}</h3>
                        <p>Total Emails</p>
                    </div>
                </div>
                <div class="stat-card">
                    <div class="stat-icon green">
                        <i class="fas fa-check-circle"></i>
                    </div>
                    <div class="stat-info">
                        <h3 id="processedEmails">
                            {{ emails|selectattr('spam', 'defined')|list|length }}
                        </h3>
                        <p>Processed</p>
                    </div>
                </div>
                <div class="stat-card">
                    <div class="stat-icon orange">
                        <i class="fas fa-bell"></i>
                    </div>
                    <div class="stat-info">
                        <h3 id="remindersCount">
                            {{ emails|selectattr('reminder', 'defined')|rejectattr('reminder', 'equalto',
                            '')|list|length }}
                        </h3>
                        <p>Reminders</p>
                    </div>
                </div>
                <div class="stat-card">
                    <div class="stat-icon red">
                        <i class="fas fa-exclamation-triangle"></i>
                    </div>
                    <div class="stat-info">
                        <h3 id="spamCount">
                            {{ emails|selectattr('spam', 'equalto', 'true')|list|length }}
                        </h3>
                        <p>Spam Detected</p>
                    </div>
                </div>
            </div>

            <!-- Filter Tabs -->
            <div class="filter-tabs">
                <button class="tab-btn active" onclick="filterByCategory('all')">All</button>
                <button class="tab-btn" onclick="filterByCategory('Work')">Work</button>
                <button class="tab-btn" onclick="filterByCategory('Personal')">Personal</button>
                <button class="tab-btn" onclick="filterByCategory('Finance')">Finance</button>
                <button class="tab-btn" onclick="filterByCategory('Promotions')">Promotions</button>
            </div>

            <!-- Email List -->
            <div class="email-list" id="emailList">
                {% for email in emails %}
                <div class="email-card" data-category="{{ email.category|default('Other') }}"
                    data-spam="{{ email.spam|default('false') }}" onclick="showEmailDetails({{ loop.index0 }})">
                    <div class="email-header">
                        <div class="email-sender">
                            <div class="avatar">{{ email.from[:1]|upper if email.from else 'U' }}</div>
                            <div class="sender-info">
                                <h4>{{ email.from|default('Unknown Sender') }}</h4>
                                <p class="email-time">{{ email.date|default('No Date') }}</p>
                            </div>
                        </div>
                        <div class="email-badges">
                            {% if email.spam == 'true' %}
                            <span class="badge badge-danger">
                                <i class="fas fa-exclamation-circle"></i> Spam
                            </span>
                            {% endif %}
                            {% if email.urgency == 'high' %}
                            <span class="badge badge-warning">
                                <i class="fas fa-bolt"></i> Urgent
                            </span>
                            {% elif email.urgency == 'moderate' %}
                            <span class="badge badge-info">
                                <i class="fas fa-clock"></i> Moderate
                            </span>
                            {% endif %}
                            {% if email.category %}
                            <span class="badge badge-category">{{ email.category }}</span>
                            {% endif %}
                        </div>
                    </div>
                    <div class="email-content">
                        <h3 class="email-subject">{{ email.subject|default('No Subject') }}</h3>
                        <p class="email-preview">{{ email.body[:200]|default('No content')|striptags }}...</p>
                    </div>
                    <div class="email-footer">
                        {% if email.sentiment %}
                        <div class="sentiment-indicator sentiment-{{ email.sentiment|lower }}">
                            <i class="fas fa-smile"></i> {{ email.sentiment }}
                        </div>
                        {% endif %}
                        {% if email.reminder and email.reminder != '' %}
                        <div class="reminder-indicator">
                            <i class="fas fa-bell"></i> Reminder set
                        </div>
                        {% endif %}
                    </div>
                </div>
                {% endfor %}
            </div>
        </main>
    </div>

    <!-- Email Detail Modal -->
    <div id="emailModal" class="modal">
        <div class="modal-content">
            <div class="modal-header">
                <h2 id="modalSubject">Email Details</h2>
                <span class="close" onclick="closeModal('emailModal')">&times;</span>
            </div>
            <div class="modal-body">
                <div class="detail-row">
                    <strong>From:</strong>
                    <span id="modalFrom"></span>
                </div>
                <div class="detail-row">
                    <strong>Date:</strong>
                    <span id="modalDate"></span>
                </div>
                <div class="detail-row">
                    <strong>Category:</strong>
                    <span id="modalCategory" class="badge"></span>
                </div>
                <div class="detail-row">
                    <strong>Sentiment:</strong>
                    <span id="modalSentiment"></span>
                </div>
                <div class="detail-row">
                    <strong>Urgency:</strong>
                    <span id="modalUrgency"></span>
                </div>
                <div class="detail-row" id="reminderRow" style="display: none;">
                    <strong>Reminder:</strong>
                    <div class="reminder-box">
                        <p id="modalReminder"></p>
                        <span id="modalReminderDate"></span>
                    </div>
                </div>
                <div class="detail-row full-width">
                    <strong>Email Body:</strong>
                    <div class="email-body-full" id="modalBody"></div>
                </div>
                <!-- AI Actions -->
                <div class="modal-footer"
                    style="padding-top: 20px; border-top: 1px solid #e2e8f0; display: flex; gap: 10px; margin-top: 20px;">
                    <button class="btn-view" onclick="summarizeEmail()"
                        style="background: linear-gradient(135deg, #a0aec0 0%, #718096 100%);">
                        <i class="fas fa-compress-alt"></i> AI Summarize
                    </button>
                    <button class="btn-view" onclick="speakEmail(event)"
                        style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);">
                        <i class="fas fa-volume-up"></i> Read Aloud
                    </button>
                    <button class="btn-view" onclick="pauseSpeech()" id="pauseBtn"
                        style="display:none; background: #ecc94b; color: #744210;">
                        <i class="fas fa-pause"></i> Pause
                    </button>
                    <button class="btn-view" onclick="resumeSpeech()" id="resumeBtn"
                        style="display:none; background: #48bb78;">
                        <i class="fas fa-play"></i> Resume
                    </button>
                    <button class="btn-view" onclick="stopSpeech()" id="stopBtn"
                        style="display:none; background: #f56565;">
                        <i class="fas fa-stop"></i> Stop
                    </button>
                </div>
                <div id="summaryBox"
                    style="display: none; margin-top: 20px; padding: 15px; background: #f7fafc; border-radius: 8px; border-left: 4px solid #667eea;">
                    <strong>AI Summary:</strong>
                    <p id="summaryText" style="margin-top: 10px; font-style: italic;"></p>
                </div>
            </div>
        </div>
    </div>

    <!-- Send Email Modal -->
    <div id="sendEmailModal" class="modal">
        <div class="modal-content">
            <div class="modal-header">
                <h2>Send Email</h2>
                <span class="close" onclick="closeModal('sendEmailModal')">&times;</span>
            </div>
            <div class="modal-body">
                <form id="sendEmailForm" onsubmit="sendEmail(event)">
                    <div class="detail-row full-width">
                        <strong>To:</strong>
                        <input type="email" id="emailRecipient" placeholder="recipient@example.com" required
                            style="width: 100%; padding: 12px; border: 1px solid #e2e8f0; border-radius: 8px;">
                    </div>
                    <div class="detail-row full-width">
                        <strong>Subject:</strong>
                        <input type="text" id="emailSubject" placeholder="Email subject" required
                            style="width: 100%; padding: 12px; border: 1px solid #e2e8f0; border-radius: 8px;">
                    </div>
                    <div class="detail-row full-width">
                        <strong>Message:</strong>
                        <textarea id="emailBody" rows="8" placeholder="Write your message here..." required
                            style="width: 100%; padding: 12px; border: 1px solid #e2e8f0; border-radius: 8px; font-family: inherit;"></textarea>
                    </div>
                    <button type="submit" class="btn-view" style="width: 100%; justify-content: center;">
                        <i class="fas fa-paper-plane"></i> Send Email
                    </button>
                </form>
            </div>
        </div>
    </div>

    <!-- AI Email Generation Modal -->
    <div id="aiEmailModal" class="modal">
        <div class="modal-content">
            <div class="modal-header" style="background: linear-gradient(135deg, #9f7aea 0%, #805ad5 100%);">
                <h2>Generate AI Email</h2>
                <span class="close" onclick="closeModal('aiEmailModal')">&times;</span>
            </div>
            <div class="modal-body">
                <form id="aiEmailForm" onsubmit="generateAIEmail(event)">
                    <div class="detail-row full-width">
                        <strong>Email Type:</strong>
                        <select id="emailType" required
                            style="width: 100%; padding: 12px; border: 1px solid #e2e8f0; border-radius: 8px;">
                            <option value="">Select email type...</option>
                            <option value="professional">Professional/Business</option>
                            <option value="followup">Follow-up</option>
                            <option value="thankyou">Thank You</option>
                            <option value="apology">Apology</option>
                            <option value="invitation">Invitation</option>
                            <option value="response">Response</option>
                            <option value="custom">Custom</option>
                        </select>
                    </div>
                    <div class="detail-row full-width">
                        <strong>Email Purpose/Details:</strong>
                        <textarea id="emailPurpose" rows="6" placeholder="Describe what the email should be about..."
                            required
                            style="width: 100%; padding: 12px; border: 1px solid #e2e8f0; border-radius: 8px; font-family: inherit;"></textarea>
                    </div>
                    <button type="submit" class="btn-view"
                        style="width: 100%; justify-content: center; background: linear-gradient(135deg, #9f7aea 0%, #805ad5 100%);">
                        <i class="fas fa-magic"></i> Generate Email
                    </button>
                </form>
                <div id="aiEmailOutput" style="display: none; margin-top: 20px;">
                    <div class="detail-row full-width">
                        <strong>Generated Email:</strong>
                        <div class="email-body-full" id="generatedEmail"></div>
                    </div>
                    <button class="btn-view" onclick="copyToClipboard()"
                        style="width: 100%; justify-content: center; margin-top: 10px;">
                        <i class="fas fa-copy"></i> Copy to Clipboard
                    </button>
                </div>
            </div>
        </div>
    </div>

    <script>
        // Store emails data
        const emailsData = {{ emails| tojson }};

        // Filter emails by search
        function filterEmails() {
            const searchTerm = document.getElementById('searchInput').value.toLowerCase();
            const emailCards = document.querySelectorAll('.email-card');

            emailCards.forEach(card => {
                const text = card.textContent.toLowerCase();
                card.style.display = text.includes(searchTerm) ? 'block' : 'none';
            });
        }

        // Filter by category
        function filterByCategory(category) {
            const emailCards = document.querySelectorAll('.email-card');
            const tabs = document.querySelectorAll('.tab-btn');

            tabs.forEach(tab => tab.classList.remove('active'));
            event.target.classList.add('active');

            emailCards.forEach(card => {
                if (category === 'all') {
                    card.style.display = 'block';
                } else {
                    const cardCategory = card.getAttribute('data-category');
                    card.style.display = cardCategory === category ? 'block' : 'none';
                }
            });
        }

        // Show email details
        function showEmailDetails(index) {
            const email = emailsData[index];
            const modal = document.getElementById('emailModal');

            document.getElementById('modalSubject').textContent = email.subject || 'No Subject';
            document.getElementById('modalFrom').textContent = email.from || 'Unknown';
            document.getElementById('modalDate').textContent = email.date || 'No Date';
            document.getElementById('modalCategory').textContent = email.category || 'Other';
            document.getElementById('modalSentiment').textContent = email.sentiment || 'N/A';
            document.getElementById('modalUrgency').textContent = email.urgency || 'N/A';
            document.getElementById('modalBody').textContent = email.body || 'No content';

            const reminderRow = document.getElementById('reminderRow');
            if (email.reminder && email.reminder !== '') {
                reminderRow.style.display = 'flex';
                document.getElementById('modalReminder').textContent = email.reminder;
                document.getElementById('modalReminderDate').textContent = email.date || '';
            } else {
                reminderRow.style.display = 'none';
            }

            modal.style.display = 'block';
        }

        // Close modal
        function closeModal(modalId) {
            document.getElementById(modalId).style.display = 'none';
            if (modalId === 'emailModal') {
                document.getElementById('summaryBox').style.display = 'none';
                window.speechSynthesis.cancel();
            }
        }

        async function summarizeEmail() {
            const body = document.getElementById('modalBody').textContent;
            const summaryBox = document.getElementById('summaryBox');
            const summaryText = document.getElementById('summaryText');

            summaryText.textContent = "Summarizing...";
            summaryBox.style.display = 'block';

            try {
                const response = await fetch('/api/summarize', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ body })
                });
                const result = await response.json();
                if (result.success) {
                    summaryText.textContent = result.summary;
                } else {
                    summaryText.textContent = "Error: " + result.message;
                }
            } catch (error) {
                summaryText.textContent = "Failed to fetch summary.";
            }
        }

        async function speakEmail(event) {
            const body = document.getElementById('modalBody').textContent;
            const speakBtn = event.currentTarget;
            const originalHTML = speakBtn.innerHTML;

            // Show loading state
            speakBtn.innerHTML = '<i class="fas fa-spinner fa-spin"></i> Processing...';
            speakBtn.disabled = true;

            try {
                const response = await fetch('/api/clean_text', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ body })
                });
                const result = await response.json();

                if (result.success) {
                    window.speechSynthesis.cancel();
                    const utterance = new SpeechSynthesisUtterance(result.cleaned_text);

                    utterance.onstart = () => {
                        document.getElementById('pauseBtn').style.display = 'flex';
                        document.getElementById('stopBtn').style.display = 'flex';
                        document.getElementById('resumeBtn').style.display = 'none';
                    };

                    utterance.onend = () => {
                        document.getElementById('pauseBtn').style.display = 'none';
                        document.getElementById('stopBtn').style.display = 'none';
                        document.getElementById('resumeBtn').style.display = 'none';
                    };

                    window.speechSynthesis.speak(utterance);
                } else {
                    alert('Error cleaning text: ' + result.message);
                }
            } catch (error) {
                console.error(error);
                alert('Failed to process text for reading.');
            } finally {
                speakBtn.innerHTML = originalHTML;
                speakBtn.disabled = false;
            }
        }

        function pauseSpeech() {
            window.speechSynthesis.pause();
            document.getElementById('pauseBtn').style.display = 'none';
            document.getElementById('resumeBtn').style.display = 'flex';
        }

        function resumeSpeech() {
            window.speechSynthesis.resume();
            document.getElementById('pauseBtn').style.display = 'flex';
            document.getElementById('resumeBtn').style.display = 'none';
        }

        function stopSpeech() {
            window.speechSynthesis.cancel();
            document.getElementById('pauseBtn').style.display = 'none';
            document.getElementById('stopBtn').style.display = 'none';
            document.getElementById('resumeBtn').style.display = 'none';
        }

        function openSendEmailModal() {
            document.getElementById('sendEmailModal').style.display = 'block';
        }

        function openAIEmailModal() {
            document.getElementById('aiEmailModal').style.display = 'block';
            document.getElementById('aiEmailOutput').style.display = 'none';
        }

        async function sendEmail(event) {
            event.preventDefault();

            const recipient = document.getElementById('emailRecipient').value;
            const subject = document.getElementById('emailSubject').value;
            const body = document.getElementById('emailBody').value;

            try {
                const response = await fetch('/api/send_email', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json',
                    },
                    body: JSON.stringify({ recipient, subject, body })
                });

                const result = await response.json();

                if (result.success) {
                    alert('Email logged successfully!');
                    closeModal('sendEmailModal');
                    document.getElementById('sendEmailForm').reset();
                } else {
                    alert('Error: ' + result.message);
                }
            } catch (error) {
                console.error('Error:', error);
                alert('Failed to send email');
            }
        }

        async function generateAIEmail(event) {
            event.preventDefault();

            const emailType = document.getElementById('emailType').value;
            const purpose = document.getElementById('emailPurpose').value;

            try {
                const response = await fetch('/api/generate_ai_email', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json',
                    },
                    body: JSON.stringify({ email_type: emailType, purpose })
                });

                const result = await response.json();

                if (result.success) {
                    document.getElementById('generatedEmail').textContent = result.email;
                    document.getElementById('aiEmailOutput').style.display = 'block';
                } else {
                    alert('Error: ' + result.message);
                }
            } catch (error) {
                console.error('Error:', error);
                alert('Failed to generate email');
            }
        }

        async function generateTodoList() {
            if (!confirm('Generate AI-enhanced to-do list from today\'s reminders?')) return;

            // Redirect to reminders page where the UI for to-do is enhanced
            window.location.href = '/reminders?generate=true';
        }

        function copyToClipboard() {
            const text = document.getElementById('generatedEmail').textContent;
            navigator.clipboard.writeText(text).then(() => {
                alert('Email copied to clipboard!');
            });
        }

        // Refresh emails
        async function refreshEmails() {
            const btn = document.querySelector('.refresh-btn');
            btn.classList.add('spinning');

            try {
                const response = await fetch('/api/refresh', { method: 'POST' });
                if (response.ok) {
                    window.location.reload();
                }
            } catch (error) {
                console.error('Error refreshing:', error);
            }

            setTimeout(() => btn.classList.remove('spinning'), 1000);
        }

        // Update last update time
        function updateLastUpdateTime() {
            fetch('/api/stats')
                .then(response => response.json())
                .then(data => {
                    document.getElementById('lastUpdate').textContent = data.last_update || '--:--';
                })
                .catch(error => console.error('Error fetching stats:', error));
        }

        // Auto-refresh stats every 30 seconds
        setInterval(updateLastUpdateTime, 30000);
        updateLastUpdateTime();

        // Close modal on outside click
        window.onclick = function (event) {
            if (event.target.classList.contains('modal')) {
                event.target.style.display = 'none';
            }
        }
    </script>
</body>

</html>

## 🛠️ Reminders Template (`templates/reminders.html`)


In [ ]:
<!DOCTYPE html>
<html lang="en">

<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Email Reminders & To-Do</title>
    <link rel="stylesheet" href="/style.css">
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
    <script src="https://cdn.jsdelivr.net/npm/marked/marked.min.js"></script>
    <style>
        /* Additional styles specific to reminders page */
        .header-btn {
            padding: 12px 20px;
            background: rgba(255, 255, 255, 0.95);
            border: none;
            border-radius: 10px;
            color: #667eea;
            font-weight: 600;
            cursor: pointer;
            display: flex;
            align-items: center;
            gap: 8px;
            transition: all 0.3s ease;
        }

        .header-btn:hover {
            transform: translateY(-2px);
            box-shadow: 0 4px 12px rgba(0, 0, 0, 0.15);
        }

        .section-title {
            color: white;
            font-size: 24px;
            margin-bottom: 20px;
        }
    </style>
</head>

<body>
    <div class="container">
        <!-- Sidebar -->
        <aside class="sidebar">
            <div class="logo">
                <i class="fas fa-envelope-open-text"></i>
                <h2>Email Manager</h2>
            </div>
            <nav class="nav-menu">
                <a href="/" class="nav-item">
                    <i class="fas fa-inbox"></i>
                    <span>Inbox</span>
                </a>
                <a href="/analysis" class="nav-item">
                    <i class="fas fa-chart-line"></i>
                    <span>Analysis</span>
                </a>
                <a href="/reminders" class="nav-item active">
                    <i class="fas fa-bell"></i>
                    <span>Reminders</span>
                </a>
                <a href="/spam" class="nav-item">
                    <i class="fas fa-shield-alt"></i>
                    <span>Spam</span>
                </a>
            </nav>
            <div class="sidebar-footer">
                <button class="refresh-btn" onclick="generateTodoList()"
                    style="background: linear-gradient(135deg, #48bb78 0%, #38a169 100%); margin-bottom: 10px;">
                    <i class="fas fa-magic"></i>
                    <span>Generate To-Do List</span>
                </button>
                <button class="refresh-btn" onclick="openSendEmailModal()">
                    <i class="fas fa-paper-plane"></i>
                    <span>Send Email</span>
                </button>
                <button class="refresh-btn" onclick="openAIEmailModal()"
                    style="background: linear-gradient(135deg, #9f7aea 0%, #805ad5 100%); margin-top: 10px;">
                    <i class="fas fa-robot"></i>
                    <span>Generate AI Email</span>
                </button>
                <button class="refresh-btn" onclick="refreshEmails()" style="margin-top: 10px;">
                    <i class="fas fa-sync-alt"></i>
                    <span>Refresh Emails</span>
                </button>
                <button class="refresh-btn" onclick="window.location.href='/logout'"
                    style="background: linear-gradient(135deg, #f56565 0%, #e53e3e 100%); margin-top: 10px;">
                    <i class="fas fa-sign-out-alt"></i>
                    <span>Logout</span>
                </button>
                <p class="last-update">Last update: <span id="lastUpdate">--:--</span></p>
            </div>
        </aside>

        <!-- Main Content -->
        <main class="main-content">
            <header class="header">
                <div class="header-left">
                    <h1>Email Reminders & To-Do</h1>
                    <p class="subtitle">Never miss an important follow-up</p>
                </div>
                <div class="header-actions" style="display: flex; gap: 10px;">
                    <button class="header-btn" onclick="filterReminders('all')" id="filterAll">
                        <i class="fas fa-list"></i> All
                    </button>
                    <button class="header-btn" onclick="filterReminders('today')" id="filterToday">
                        <i class="fas fa-calendar-day"></i> Today
                    </button>
                    <button class="header-btn" onclick="filterReminders('urgent')" id="filterUrgent">
                        <i class="fas fa-exclamation-circle"></i> Urgent
                    </button>
                </div>
            </header>

            <!-- Reminder Stats -->
            <div class="stats-grid">
                <div class="stat-card">
                    <div class="stat-icon orange">
                        <i class="fas fa-bell"></i>
                    </div>
                    <div class="stat-info">
                        <h3>{{ reminders|length }}</h3>
                        <p>Total Reminders</p>
                    </div>
                </div>
                <div class="stat-card">
                    <div class="stat-icon red">
                        <i class="fas fa-exclamation-circle"></i>
                    </div>
                    <div class="stat-info">
                        <h3>{{ urgent_reminders }}</h3>
                        <p>Urgent</p>
                    </div>
                </div>
                <div class="stat-card">
                    <div class="stat-icon blue">
                        <i class="fas fa-calendar-check"></i>
                    </div>
                    <div class="stat-info">
                        <h3 id="todayCount">{{ upcoming_reminders }}</h3>
                        <p>Today's Tasks</p>
                    </div>
                </div>
            </div>

            <!-- Reminders List -->
            <div class="reminders-container">
                <h2 class="section-title">Active Reminders</h2>

                {% if reminders|length == 0 %}
                <div class="empty-state">
                    <i class="fas fa-bell-slash"></i>
                    <h3>No Reminders Found</h3>
                    <p>You don't have any reminders set up yet.</p>
                </div>
                {% else %}
                <div class="reminder-list" id="reminderList">
                    {% for email in reminders %}
                    <div class="reminder-card {% if email.urgency == 'high' %}urgent{% endif %}"
                        data-urgency="{{ email.urgency }}" data-date="{{ email.reminder_date }}">
                        <div class="reminder-header">
                            <div class="reminder-icon">
                                <i class="fas fa-bell"></i>
                            </div>
                            <div class="reminder-details">
                                <h3>{{ email.reminder }}</h3>
                                <p class="reminder-date">
                                    <i class="fas fa-calendar"></i>
                                    {% if email.reminder_date and email.reminder_date != 'none' %}
                                    Due: {{ email.reminder_date }}
                                    {% else %}
                                    No due date set
                                    {% endif %}
                                </p>
                            </div>
                            {% if email.urgency == 'high' %}
                            <span class="urgent-badge">
                                <i class="fas fa-bolt"></i> URGENT
                            </span>
                            {% endif %}
                        </div>

                        <div class="reminder-body">
                            <div class="email-info">
                                <div class="info-item">
                                    <strong>From:</strong>
                                    <span>{{ email.from }}</span>
                                </div>
                                <div class="info-item">
                                    <strong>Subject:</strong>
                                    <span>{{ email.subject }}</span>
                                </div>
                                <div class="info-item">
                                    <strong>Category:</strong>
                                    <span class="badge badge-category">{{ email.category|default('Other') }}</span>
                                </div>
                            </div>

                            <div class="reminder-preview">
                                {{ email.body[:150]|default('No preview available') }}...
                            </div>
                        </div>

                        <div class="reminder-footer">
                            <button class="btn-view"
                                onclick="viewEmailDetails('{{ email.subject }}', '{{ email.from }}', '{{ email.body }}')">
                                <i class="fas fa-eye"></i> View Full Email
                            </button>
                            <button class="btn-view" onclick="markComplete('{{ email.id }}')"
                                style="background: linear-gradient(135deg, #48bb78 0%, #38a169 100%); margin-left: 10px;">
                                <i class="fas fa-check"></i> Mark Complete
                            </button>
                        </div>
                    </div>
                    {% endfor %}
                </div>
                {% endif %}
            </div>
        </main>
    </div>

    <!-- Email Detail Modal -->
    <div id="emailModal" class="modal">
        <div class="modal-content">
            <div class="modal-header">
                <h2 id="modalSubject">Email Details</h2>
                <span class="close" onclick="closeModal('emailModal')">&times;</span>
            </div>
            <div class="modal-body">
                <div class="detail-row">
                    <strong>From:</strong>
                    <span id="modalFrom"></span>
                </div>
                <div class="detail-row full-width">
                    <strong>Full Message:</strong>
                    <div class="email-body-full" id="modalBody"></div>
                </div>
                <!-- AI Actions -->
                <div class="modal-footer"
                    style="padding-top: 20px; border-top: 1px solid #e2e8f0; display: flex; gap: 10px; margin-top: 20px;">
                    <button class="btn-view" onclick="summarizeEmail()"
                        style="background: linear-gradient(135deg, #a0aec0 0%, #718096 100%);">
                        <i class="fas fa-compress-alt"></i> AI Summarize
                    </button>
                    <button class="btn-view" onclick="speakEmail(event)"
                        style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);">
                        <i class="fas fa-volume-up"></i> Read Aloud
                    </button>
                    <button class="btn-view" onclick="pauseSpeech()" id="pauseBtn"
                        style="display:none; background: #ecc94b; color: #744210;">
                        <i class="fas fa-pause"></i> Pause
                    </button>
                    <button class="btn-view" onclick="resumeSpeech()" id="resumeBtn"
                        style="display:none; background: #48bb78;">
                        <i class="fas fa-play"></i> Resume
                    </button>
                    <button class="btn-view" onclick="stopSpeech()" id="stopBtn"
                        style="display:none; background: #f56565;">
                        <i class="fas fa-stop"></i> Stop
                    </button>
                </div>
                <div id="summaryBox"
                    style="display: none; margin-top: 20px; padding: 15px; background: #f7fafc; border-radius: 8px; border-left: 4px solid #667eea;">
                    <strong>AI Summary:</strong>
                    <p id="summaryText" style="margin-top: 10px; font-style: italic;"></p>
                </div>
            </div>
        </div>
    </div>

    <!-- Send Email Modal -->
    <div id="sendEmailModal" class="modal">
        <div class="modal-content">
            <div class="modal-header">
                <h2>Send Email</h2>
                <span class="close" onclick="closeModal('sendEmailModal')">&times;</span>
            </div>
            <div class="modal-body">
                <form id="sendEmailForm" onsubmit="sendEmail(event)">
                    <div class="detail-row full-width">
                        <strong>To:</strong>
                        <input type="email" id="emailRecipient" class="search-box input"
                            placeholder="recipient@example.com" required
                            style="width: 100%; padding: 12px; border: 1px solid #e2e8f0; border-radius: 8px;">
                    </div>
                    <div class="detail-row full-width">
                        <strong>Subject:</strong>
                        <input type="text" id="emailSubject" class="search-box input" placeholder="Email subject"
                            required style="width: 100%; padding: 12px; border: 1px solid #e2e8f0; border-radius: 8px;">
                    </div>
                    <div class="detail-row full-width">
                        <strong>Message:</strong>
                        <textarea id="emailBody" rows="8" placeholder="Write your message here..." required
                            style="width: 100%; padding: 12px; border: 1px solid #e2e8f0; border-radius: 8px; font-family: inherit;"></textarea>
                    </div>
                    <button type="submit" class="btn-view" style="width: 100%; justify-content: center;">
                        <i class="fas fa-paper-plane"></i> Send Email
                    </button>
                </form>
            </div>
        </div>
    </div>

    <!-- AI Email Generation Modal -->
    <div id="aiEmailModal" class="modal">
        <div class="modal-content">
            <div class="modal-header" style="background: linear-gradient(135deg, #9f7aea 0%, #805ad5 100%);">
                <h2>Generate AI Email</h2>
                <span class="close" onclick="closeModal('aiEmailModal')">&times;</span>
            </div>
            <div class="modal-body">
                <form id="aiEmailForm" onsubmit="generateAIEmail(event)">
                    <div class="detail-row full-width">
                        <strong>Email Type:</strong>
                        <select id="emailType" class="search-box input" required
                            style="width: 100%; padding: 12px; border: 1px solid #e2e8f0; border-radius: 8px;">
                            <option value="">Select email type...</option>
                            <option value="professional">Professional/Business</option>
                            <option value="followup">Follow-up</option>
                            <option value="thankyou">Thank You</option>
                            <option value="apology">Apology</option>
                            <option value="invitation">Invitation</option>
                            <option value="response">Response</option>
                            <option value="custom">Custom</option>
                        </select>
                    </div>
                    <div class="detail-row full-width">
                        <strong>Email Purpose/Details:</strong>
                        <textarea id="emailPurpose" rows="6" placeholder="Describe what the email should be about..."
                            required
                            style="width: 100%; padding: 12px; border: 1px solid #e2e8f0; border-radius: 8px; font-family: inherit;"></textarea>
                    </div>
                    <button type="submit" class="btn-view"
                        style="width: 100%; justify-content: center; background: linear-gradient(135deg, #9f7aea 0%, #805ad5 100%);">
                        <i class="fas fa-magic"></i> Generate Email
                    </button>
                </form>
                <div id="aiEmailOutput" style="display: none; margin-top: 20px;">
                    <div class="detail-row full-width">
                        <strong>Generated Email:</strong>
                        <div class="email-body-full" id="generatedEmail"></div>
                    </div>
                    <button class="btn-view" onclick="copyToClipboard()"
                        style="width: 100%; justify-content: center; margin-top: 10px;">
                        <i class="fas fa-copy"></i> Copy to Clipboard
                    </button>
                </div>
            </div>
        </div>
    </div>

    <!-- AI To-Do Digest Modal -->
    <div id="todoModal" class="modal">
        <div class="modal-content" style="max-width: 900px;">
            <div class="modal-header" style="background: linear-gradient(135deg, #48bb78 0%, #38a169 100%);">
                <h2>AI To-Do Digest</h2>
                <span class="close" onclick="closeModal('todoModal')">&times;</span>
            </div>
            <div class="modal-body">
                <div class="todo-results">
                    <div class="ai-suggestion-box">
                        <h3><i class="fas fa-magic"></i> AI Prioritized Plan</h3>
                        <div id="aiTodoContent" class="markdown-body"></div>
                    </div>

                    <div class="task-table-container">
                        <h3><i class="fas fa-tasks"></i> Task Breakdown</h3>
                        <div style="overflow-x: auto;">
                            <table class="task-table">
                                <thead>
                                    <tr>
                                        <th>Task</th>
                                        <th>Urgency</th>
                                        <th>Category</th>
                                        <th>Source</th>
                                    </tr>
                                </thead>
                                <tbody id="todoTaskList"></tbody>
                            </table>
                        </div>
                    </div>
                </div>
            </div>
        </div>
    </div>

    <script>
        function viewEmailDetails(subject, from, body) {
            document.getElementById('modalSubject').textContent = subject;
            document.getElementById('modalFrom').textContent = from;
            document.getElementById('modalBody').textContent = body;
            document.getElementById('emailModal').style.display = 'block';
        }

        function closeModal(modalId) {
            document.getElementById(modalId).style.display = 'none';
            if (modalId === 'emailModal') {
                document.getElementById('summaryBox').style.display = 'none';
                window.speechSynthesis.cancel();
            }
        }

        async function summarizeEmail() {
            const body = document.getElementById('modalBody').textContent;
            const summaryBox = document.getElementById('summaryBox');
            const summaryText = document.getElementById('summaryText');

            summaryText.textContent = "Summarizing...";
            summaryBox.style.display = 'block';

            try {
                const response = await fetch('/api/summarize', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ body })
                });
                const result = await response.json();
                if (result.success) {
                    summaryText.textContent = result.summary;
                } else {
                    summaryText.textContent = "Error: " + result.message;
                }
            } catch (error) {
                summaryText.textContent = "Failed to fetch summary.";
            }
        }

        async function speakEmail(event) {
            const body = document.getElementById('modalBody').textContent;
            const speakBtn = event.currentTarget;
            const originalHTML = speakBtn.innerHTML;

            // Show loading state
            speakBtn.innerHTML = '<i class="fas fa-spinner fa-spin"></i> Processing...';
            speakBtn.disabled = true;

            try {
                const response = await fetch('/api/clean_text', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ body })
                });
                const result = await response.json();

                if (result.success) {
                    window.speechSynthesis.cancel();
                    const utterance = new SpeechSynthesisUtterance(result.cleaned_text);

                    utterance.onstart = () => {
                        document.getElementById('pauseBtn').style.display = 'flex';
                        document.getElementById('stopBtn').style.display = 'flex';
                        document.getElementById('resumeBtn').style.display = 'none';
                    };

                    utterance.onend = () => {
                        document.getElementById('pauseBtn').style.display = 'none';
                        document.getElementById('stopBtn').style.display = 'none';
                        document.getElementById('resumeBtn').style.display = 'none';
                    };

                    window.speechSynthesis.speak(utterance);
                } else {
                    alert('Error cleaning text: ' + result.message);
                }
            } catch (error) {
                console.error(error);
                alert('Failed to process text for reading.');
            } finally {
                speakBtn.innerHTML = originalHTML;
                speakBtn.disabled = false;
            }
        }

        function pauseSpeech() {
            window.speechSynthesis.pause();
            document.getElementById('pauseBtn').style.display = 'none';
            document.getElementById('resumeBtn').style.display = 'flex';
        }

        function resumeSpeech() {
            window.speechSynthesis.resume();
            document.getElementById('pauseBtn').style.display = 'flex';
            document.getElementById('resumeBtn').style.display = 'none';
        }

        function stopSpeech() {
            window.speechSynthesis.cancel();
            document.getElementById('pauseBtn').style.display = 'none';
            document.getElementById('stopBtn').style.display = 'none';
            document.getElementById('resumeBtn').style.display = 'none';
        }

        function openSendEmailModal() {
            document.getElementById('sendEmailModal').style.display = 'block';
        }

        function openAIEmailModal() {
            document.getElementById('aiEmailModal').style.display = 'block';
            document.getElementById('aiEmailOutput').style.display = 'none';
        }

        async function sendEmail(event) {
            event.preventDefault();

            const recipient = document.getElementById('emailRecipient').value;
            const subject = document.getElementById('emailSubject').value;
            const body = document.getElementById('emailBody').value;

            try {
                const response = await fetch('/api/send_email', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json',
                    },
                    body: JSON.stringify({ recipient, subject, body })
                });

                const result = await response.json();

                if (result.success) {
                    alert('Email logged successfully!');
                    closeModal('sendEmailModal');
                    document.getElementById('sendEmailForm').reset();
                } else {
                    alert('Error: ' + result.message);
                }
            } catch (error) {
                console.error('Error:', error);
                alert('Failed to send email');
            }
        }

        async function generateAIEmail(event) {
            event.preventDefault();

            const emailType = document.getElementById('emailType').value;
            const purpose = document.getElementById('emailPurpose').value;

            try {
                const response = await fetch('/api/generate_ai_email', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json',
                    },
                    body: JSON.stringify({ email_type: emailType, purpose })
                });

                const result = await response.json();

                if (result.success) {
                    document.getElementById('generatedEmail').textContent = result.email;
                    document.getElementById('aiEmailOutput').style.display = 'block';
                } else {
                    alert('Error: ' + result.message);
                }
            } catch (error) {
                console.error('Error:', error);
                alert('Failed to generate email');
            }
        }

        async function generateTodoList() {
            if (!confirm('Generate AI-enhanced to-do list from today\'s reminders?')) return;

            const btn = event.currentTarget || document.querySelector('button[onclick="generateTodoList()"]');
            const originalText = btn.innerHTML;
            btn.innerHTML = '<i class="fas fa-spinner fa-spin"></i> <span>Generating...</span>';
            btn.disabled = true;

            try {
                const response = await fetch('/api/generate_todo', {
                    method: 'POST'
                });

                const result = await response.json();

                if (result.success) {
                    // Update the UI with the results
                    document.getElementById('todoModal').style.display = 'block';
                    document.getElementById('aiTodoContent').innerHTML = marked.parse(result.ai_todo);

                    // Populate task list
                    const taskList = document.getElementById('todoTaskList');
                    taskList.innerHTML = '';
                    result.tasks.forEach(task => {
                        const tr = document.createElement('tr');
                        tr.innerHTML = `
                            <td>${task.Task}</td>
                            <td><span class="badge ${task.Urgency === 'high' ? 'badge-urgent' : 'badge-low'}">${task.Urgency}</span></td>
                            <td>${task.Category}</td>
                            <td>${task.Source}</td>
                        `;
                        taskList.appendChild(tr);
                    });
                } else {
                    alert('Error: ' + result.message);
                }
            } catch (error) {
                console.error('Error:', error);
                alert('Failed to generate to-do list');
            } finally {
                btn.innerHTML = originalText;
                btn.disabled = false;
            }
        }

        function copyToClipboard() {
            const text = document.getElementById('generatedEmail').textContent;
            navigator.clipboard.writeText(text).then(() => {
                alert('Email copied to clipboard!');
            });
        }

        // Refresh emails
        async function refreshEmails() {
            const btn = document.querySelector('.refresh-btn');
            btn.classList.add('spinning');

            try {
                const response = await fetch('/api/refresh', { method: 'POST' });
                if (response.ok) {
                    window.location.reload();
                }
            } catch (error) {
                console.error('Error refreshing:', error);
            }

            setTimeout(() => btn.classList.remove('spinning'), 1000);
        }

        // Update last update time
        function updateLastUpdateTime() {
            fetch('/api/stats')
                .then(response => response.json())
                .then(data => {
                    const el = document.getElementById('lastUpdate');
                    if (el) el.textContent = data.last_update || '--:--';
                })
                .catch(error => console.error('Error fetching stats:', error));
        }

        // Auto-refresh stats every 30 seconds
        setInterval(updateLastUpdateTime, 30000);
        updateLastUpdateTime();

        async function markComplete(emailId) {
            if (!confirm('Mark this reminder as complete?')) return;

            try {
                const response = await fetch('/api/mark_complete', {
                    method: 'POST',
                    headers: {
                        'Content-Type': 'application/json',
                    },
                    body: JSON.stringify({ email_id: emailId })
                });

                const result = await response.json();

                if (result.success) {
                    alert('Reminder marked as complete!');
                    window.location.reload();
                } else {
                    alert('Error: ' + result.message);
                }
            } catch (error) {
                console.error('Error:', error);
                alert('Failed to mark complete');
            }
        }

        function filterReminders(type) {
            const cards = document.querySelectorAll('.reminder-card');
            const today = new Date().toISOString().split('T')[0];
            let count = 0;

            // Reset button styles
            document.querySelectorAll('.header-btn').forEach(btn => {
                btn.style.background = 'rgba(255, 255, 255, 0.95)';
                btn.style.color = '#667eea';
            });

            // Highlight active button
            const activeBtn = document.getElementById('filter' + type.charAt(0).toUpperCase() + type.slice(1));
            if (activeBtn) {
                activeBtn.style.background = '#667eea';
                activeBtn.style.color = 'white';
            }

            cards.forEach(card => {
                const urgency = card.getAttribute('data-urgency');
                const date = card.getAttribute('data-date');

                let show = false;

                if (type === 'all') {
                    show = true;
                } else if (type === 'today') {
                    show = date && date.includes(today);
                } else if (type === 'urgent') {
                    show = urgency === 'high';
                }

                card.style.display = show ? 'block' : 'none';
                if (show) count++;
            });

            // Update today count if filtering by today
            if (type === 'today') {
                document.getElementById('todayCount').textContent = count;
            }
        }

        // Close modals on outside click
        window.onclick = function (event) {
            if (event.target.classList.contains('modal')) {
                event.target.style.display = 'none';
            }
        }
        // Check for auto-generate parameter
        window.onload = function () {
            const urlParams = new URLSearchParams(window.location.search);
            if (urlParams.get('generate') === 'true') {
                generateTodoList();
            }
        };
    </script>

    <style>
        .header-btn {
            padding: 12px 20px;
            background: rgba(255, 255, 255, 0.95);
            border: none;
            border-radius: 10px;
            color: #667eea;
            font-weight: 600;
            cursor: pointer;
            display: flex;
            align-items: center;
            gap: 8px;
            transition: all 0.3s ease;
        }

        .header-btn:hover {
            transform: translateY(-2px);
            box-shadow: 0 4px 12px rgba(0, 0, 0, 0.15);
        }

        .section-title {
            color: white;
            font-size: 24px;
            margin-bottom: 20px;
        }
    </style>
</body>

</html>

## 🛠️ Login Template (`templates/login.html`)


In [ ]:
<!DOCTYPE html>
<html lang="en">

<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Login - Email Manager</title>
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            display: flex;
            align-items: center;
            justify-content: center;
            padding: 20px;
        }

        .login-container {
            background: rgba(255, 255, 255, 0.95);
            backdrop-filter: blur(10px);
            border-radius: 20px;
            box-shadow: 0 20px 60px rgba(0, 0, 0, 0.3);
            overflow: hidden;
            width: 100%;
            max-width: 450px;
            animation: slideUp 0.5s ease;
        }

        @keyframes slideUp {
            from {
                transform: translateY(50px);
                opacity: 0;
            }

            to {
                transform: translateY(0);
                opacity: 1;
            }
        }

        .login-header {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            padding: 40px 30px;
            text-align: center;
            color: white;
        }

        .login-header i {
            font-size: 60px;
            margin-bottom: 15px;
            animation: bounce 2s infinite;
        }

        @keyframes bounce {

            0%,
            100% {
                transform: translateY(0);
            }

            50% {
                transform: translateY(-10px);
            }
        }

        .login-header h1 {
            font-size: 28px;
            margin-bottom: 8px;
        }

        .login-header p {
            font-size: 14px;
            opacity: 0.9;
        }

        .login-body {
            padding: 40px 30px;
        }

        .error-message {
            background: #fee;
            border-left: 4px solid #f56565;
            color: #742a2a;
            padding: 12px 15px;
            border-radius: 8px;
            margin-bottom: 20px;
            display: flex;
            align-items: center;
            gap: 10px;
            animation: shake 0.5s;
        }

        @keyframes shake {

            0%,
            100% {
                transform: translateX(0);
            }

            25% {
                transform: translateX(-10px);
            }

            75% {
                transform: translateX(10px);
            }
        }

        .error-message i {
            font-size: 18px;
        }

        .form-group {
            margin-bottom: 25px;
            position: relative;
        }

        .form-group label {
            display: block;
            margin-bottom: 8px;
            color: #4a5568;
            font-weight: 600;
            font-size: 14px;
        }

        .input-wrapper {
            position: relative;
        }

        .input-wrapper i {
            position: absolute;
            left: 15px;
            top: 50%;
            transform: translateY(-50%);
            color: #a0aec0;
            font-size: 18px;
        }

        .form-control {
            width: 100%;
            padding: 14px 15px 14px 45px;
            border: 2px solid #e2e8f0;
            border-radius: 10px;
            font-size: 15px;
            transition: all 0.3s ease;
            font-family: inherit;
        }

        .form-control:focus {
            outline: none;
            border-color: #667eea;
            box-shadow: 0 0 0 4px rgba(102, 126, 234, 0.1);
        }

        .form-control::placeholder {
            color: #cbd5e0;
        }

        .remember-forgot {
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 25px;
            font-size: 14px;
        }

        .remember-me {
            display: flex;
            align-items: center;
            gap: 8px;
            color: #4a5568;
            cursor: pointer;
        }

        .remember-me input[type="checkbox"] {
            width: 18px;
            height: 18px;
            cursor: pointer;
        }

        .forgot-password {
            color: #667eea;
            text-decoration: none;
            font-weight: 600;
            transition: all 0.3s ease;
        }

        .forgot-password:hover {
            color: #5568d3;
            text-decoration: underline;
        }

        .login-btn {
            width: 100%;
            padding: 14px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            border: none;
            border-radius: 10px;
            font-size: 16px;
            font-weight: 600;
            cursor: pointer;
            transition: all 0.3s ease;
            display: flex;
            align-items: center;
            justify-content: center;
            gap: 10px;
        }

        .login-btn:hover {
            transform: translateY(-2px);
            box-shadow: 0 8px 20px rgba(102, 126, 234, 0.4);
        }

        .login-btn:active {
            transform: translateY(0);
        }

        .login-btn i {
            font-size: 18px;
        }

        .divider {
            display: flex;
            align-items: center;
            margin: 30px 0;
            color: #a0aec0;
            font-size: 14px;
        }

        .divider::before,
        .divider::after {
            content: '';
            flex: 1;
            height: 1px;
            background: #e2e8f0;
        }

        .divider span {
            padding: 0 15px;
        }

        .login-footer {
            text-align: center;
            padding: 20px;
            background: #f7fafc;
            border-top: 1px solid #e2e8f0;
            font-size: 14px;
            color: #718096;
        }

        .login-footer a {
            color: #667eea;
            text-decoration: none;
            font-weight: 600;
        }

        .login-footer a:hover {
            text-decoration: underline;
        }

        /* Password visibility toggle */
        .password-toggle {
            position: absolute;
            right: 15px;
            top: 50%;
            transform: translateY(-50%);
            cursor: pointer;
            color: #a0aec0;
            transition: all 0.3s ease;
        }

        .password-toggle:hover {
            color: #667eea;
        }

        @media (max-width: 480px) {
            .login-header {
                padding: 30px 20px;
            }

            .login-body {
                padding: 30px 20px;
            }

            .login-header h1 {
                font-size: 24px;
            }

            .remember-forgot {
                flex-direction: column;
                gap: 10px;
                align-items: flex-start;
            }
        }
    </style>
</head>

<body>
    <div class="login-container">
        <div class="login-header">
            <i class="fas fa-envelope-open-text"></i>
            <h1>Email Manager</h1>
            <p>Sign in to manage your emails</p>
        </div>

        <div class="login-body">
            {% if error %}
            <div class="error-message">
                <i class="fas fa-exclamation-circle"></i>
                <span>{{ error }}</span>
            </div>
            {% endif %}

            <form method="POST" action="/login" id="loginForm">
                <div class="form-group">
                    <label for="username">Username</label>
                    <div class="input-wrapper">
                        <i class="fas fa-user"></i>
                        <input type="text" id="username" name="username" class="form-control"
                            placeholder="Enter your username" required autofocus>
                    </div>
                </div>

                <div class="form-group">
                    <label for="password">Password</label>
                    <div class="input-wrapper">
                        <i class="fas fa-lock"></i>
                        <input type="password" id="password" name="password" class="form-control"
                            placeholder="Enter your password" required>
                        <i class="fas fa-eye password-toggle" onclick="togglePassword()"></i>
                    </div>
                </div>

                <div class="remember-forgot">
                    <label class="remember-me">
                        <input type="checkbox" name="remember">
                        <span>Remember me</span>
                    </label>
                    <a href="#" class="forgot-password">Forgot password?</a>
                </div>

                <button type="submit" class="login-btn">
                    <i class="fas fa-sign-in-alt"></i>
                    <span>Sign In</span>
                </button>
            </form>
        </div>

        <div class="login-footer">
            <p>Don't have an account? <a href="/signup">Sign Up</a></p>
        </div>
    </div>

    <script>
        function togglePassword() {
            const passwordInput = document.getElementById('password');
            const toggleIcon = document.querySelector('.password-toggle');

            if (passwordInput.type === 'password') {
                passwordInput.type = 'text';
                toggleIcon.classList.remove('fa-eye');
                toggleIcon.classList.add('fa-eye-slash');
            } else {
                passwordInput.type = 'password';
                toggleIcon.classList.remove('fa-eye-slash');
                toggleIcon.classList.add('fa-eye');
            }
        }

        // Auto-focus on username field
        document.addEventListener('DOMContentLoaded', function () {
            document.getElementById('username').focus();
        });

        // Handle form submission with loading state
        document.getElementById('loginForm').addEventListener('submit', function (e) {
            const btn = this.querySelector('.login-btn');
            const icon = btn.querySelector('i');
            const span = btn.querySelector('span');

            btn.disabled = true;
            icon.classList.remove('fa-sign-in-alt');
            icon.classList.add('fa-spinner', 'fa-spin');
            span.textContent = 'Signing In...';
        });
    </script>
</body>

</html>

## 🛠️ Signup Template (`templates/signup.html`)


In [ ]:
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Sign Up - Email Manager</title>
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            display: flex;
            align-items: center;
            justify-content: center;
            padding: 20px;
        }

        .signup-container {
            background: rgba(255, 255, 255, 0.95);
            backdrop-filter: blur(10px);
            border-radius: 20px;
            box-shadow: 0 20px 60px rgba(0, 0, 0, 0.3);
            overflow: hidden;
            width: 100%;
            max-width: 450px;
            animation: slideUp 0.5s ease;
        }

        @keyframes slideUp {
            from {
                transform: translateY(50px);
                opacity: 0;
            }
            to {
                transform: translateY(0);
                opacity: 1;
            }
        }

        .signup-header {
            background: linear-gradient(135deg, #48bb78 0%, #38a169 100%);
            padding: 40px 30px;
            text-align: center;
            color: white;
        }

        .signup-header i {
            font-size: 60px;
            margin-bottom: 15px;
            animation: bounce 2s infinite;
        }

        @keyframes bounce {
            0%, 100% { transform: translateY(0); }
            50% { transform: translateY(-10px); }
        }

        .signup-header h1 {
            font-size: 28px;
            margin-bottom: 8px;
        }

        .signup-header p {
            font-size: 14px;
            opacity: 0.9;
        }

        .signup-body {
            padding: 40px 30px;
        }

        .error-message {
            background: #fee;
            border-left: 4px solid #f56565;
            color: #742a2a;
            padding: 12px 15px;
            border-radius: 8px;
            margin-bottom: 20px;
            display: flex;
            align-items: center;
            gap: 10px;
            animation: shake 0.5s;
        }

        @keyframes shake {
            0%, 100% { transform: translateX(0); }
            25% { transform: translateX(-10px); }
            75% { transform: translateX(10px); }
        }

        .error-message i {
            font-size: 18px;
        }

        .success-message {
            background: #e6ffed;
            border-left: 4px solid #48bb78;
            color: #22543d;
            padding: 12px 15px;
            border-radius: 8px;
            margin-bottom: 20px;
            display: flex;
            align-items: center;
            gap: 10px;
        }

        .form-group {
            margin-bottom: 25px;
            position: relative;
        }

        .form-group label {
            display: block;
            margin-bottom: 8px;
            color: #4a5568;
            font-weight: 600;
            font-size: 14px;
        }

        .input-wrapper {
            position: relative;
        }

        .input-wrapper i {
            position: absolute;
            left: 15px;
            top: 50%;
            transform: translateY(-50%);
            color: #a0aec0;
            font-size: 18px;
        }

        .form-control {
            width: 100%;
            padding: 14px 15px 14px 45px;
            border: 2px solid #e2e8f0;
            border-radius: 10px;
            font-size: 15px;
            transition: all 0.3s ease;
            font-family: inherit;
        }

        .form-control:focus {
            outline: none;
            border-color: #48bb78;
            box-shadow: 0 0 0 4px rgba(72, 187, 120, 0.1);
        }

        .form-control::placeholder {
            color: #cbd5e0;
        }

        .password-toggle {
            position: absolute;
            right: 15px;
            top: 50%;
            transform: translateY(-50%);
            cursor: pointer;
            color: #a0aec0;
            transition: all 0.3s ease;
        }

        .password-toggle:hover {
            color: #48bb78;
        }

        .password-requirements {
            background: #f7fafc;
            border-radius: 8px;
            padding: 12px 15px;
            margin-top: 10px;
            font-size: 13px;
        }

        .password-requirements p {
            color: #4a5568;
            font-weight: 600;
            margin-bottom: 8px;
        }

        .password-requirements ul {
            list-style: none;
            padding: 0;
        }

        .password-requirements li {
            color: #718096;
            padding: 4px 0;
            display: flex;
            align-items: center;
            gap: 8px;
        }

        .password-requirements li i {
            font-size: 12px;
            color: #cbd5e0;
        }

        .password-requirements li.valid i {
            color: #48bb78;
        }

        .signup-btn {
            width: 100%;
            padding: 14px;
            background: linear-gradient(135deg, #48bb78 0%, #38a169 100%);
            color: white;
            border: none;
            border-radius: 10px;
            font-size: 16px;
            font-weight: 600;
            cursor: pointer;
            transition: all 0.3s ease;
            display: flex;
            align-items: center;
            justify-content: center;
            gap: 10px;
        }

        .signup-btn:hover {
            transform: translateY(-2px);
            box-shadow: 0 8px 20px rgba(72, 187, 120, 0.4);
        }

        .signup-btn:active {
            transform: translateY(0);
        }

        .signup-btn i {
            font-size: 18px;
        }

        .signup-footer {
            text-align: center;
            padding: 20px;
            background: #f7fafc;
            border-top: 1px solid #e2e8f0;
            font-size: 14px;
            color: #718096;
        }

        .signup-footer a {
            color: #667eea;
            text-decoration: none;
            font-weight: 600;
        }

        .signup-footer a:hover {
            text-decoration: underline;
        }

        @media (max-width: 480px) {
            .signup-header {
                padding: 30px 20px;
            }

            .signup-body {
                padding: 30px 20px;
            }

            .signup-header h1 {
                font-size: 24px;
            }
        }
    </style>
</head>
<body>
    <div class="signup-container">
        <div class="signup-header">
            <i class="fas fa-user-plus"></i>
            <h1>Create Account</h1>
            <p>Join Email Manager today</p>
        </div>

        <div class="signup-body">
            {% if error %}
            <div class="error-message">
                <i class="fas fa-exclamation-circle"></i>
                <span>{{ error }}</span>
            </div>
            {% endif %}

            <form method="POST" action="/signup" id="signupForm">
                <div class="form-group">
                    <label for="username">Username</label>
                    <div class="input-wrapper">
                        <i class="fas fa-user"></i>
                        <input type="text" 
                               id="username" 
                               name="username" 
                               class="form-control" 
                               placeholder="Choose a username" 
                               required 
                               minlength="3"
                               autofocus>
                    </div>
                </div>

                <div class="form-group">
                    <label for="password">Password</label>
                    <div class="input-wrapper">
                        <i class="fas fa-lock"></i>
                        <input type="password" 
                               id="password" 
                               name="password" 
                               class="form-control" 
                               placeholder="Create a password" 
                               required
                               minlength="6"
                               oninput="checkPassword()">
                        <i class="fas fa-eye password-toggle" onclick="togglePassword('password')"></i>
                    </div>
                    <div class="password-requirements">
                        <p>Password Requirements:</p>
                        <ul id="requirements">
                            <li id="req-length"><i class="fas fa-circle"></i> At least 6 characters</li>
                        </ul>
                    </div>
                </div>

                <div class="form-group">
                    <label for="confirm_password">Confirm Password</label>
                    <div class="input-wrapper">
                        <i class="fas fa-lock"></i>
                        <input type="password" 
                               id="confirm_password" 
                               name="confirm_password" 
                               class="form-control" 
                               placeholder="Confirm your password" 
                               required
                               oninput="checkPasswordMatch()">
                        <i class="fas fa-eye password-toggle" onclick="togglePassword('confirm_password')"></i>
                    </div>
                    <p id="match-message" style="margin-top: 8px; font-size: 13px;"></p>
                </div>

                <button type="submit" class="signup-btn" id="submitBtn" disabled>
                    <i class="fas fa-user-plus"></i>
                    <span>Create Account</span>
                </button>
            </form>
        </div>

        <div class="signup-footer">
            <p>Already have an account? <a href="/login">Sign In</a></p>
        </div>
    </div>

    <script>
        function togglePassword(fieldId) {
            const passwordInput = document.getElementById(fieldId);
            const toggleIcon = passwordInput.nextElementSibling;
            
            if (passwordInput.type === 'password') {
                passwordInput.type = 'text';
                toggleIcon.classList.remove('fa-eye');
                toggleIcon.classList.add('fa-eye-slash');
            } else {
                passwordInput.type = 'password';
                toggleIcon.classList.remove('fa-eye-slash');
                toggleIcon.classList.add('fa-eye');
            }
        }

        function checkPassword() {
            const password = document.getElementById('password').value;
            const lengthReq = document.getElementById('req-length');
            
            // Check length
            if (password.length >= 6) {
                lengthReq.classList.add('valid');
                lengthReq.querySelector('i').classList.remove('fa-circle');
                lengthReq.querySelector('i').classList.add('fa-check-circle');
            } else {
                lengthReq.classList.remove('valid');
                lengthReq.querySelector('i').classList.remove('fa-check-circle');
                lengthReq.querySelector('i').classList.add('fa-circle');
            }
            
            checkPasswordMatch();
            updateSubmitButton();
        }

        function checkPasswordMatch() {
            const password = document.getElementById('password').value;
            const confirmPassword = document.getElementById('confirm_password').value;
            const matchMessage = document.getElementById('match-message');
            
            if (confirmPassword.length === 0) {
                matchMessage.textContent = '';
                matchMessage.style.color = '';
            } else if (password === confirmPassword) {
                matchMessage.textContent = '✓ Passwords match';
                matchMessage.style.color = '#48bb78';
            } else {
                matchMessage.textContent = '✗ Passwords do not match';
                matchMessage.style.color = '#f56565';
            }
            
            updateSubmitButton();
        }

        function updateSubmitButton() {
            const password = document.getElementById('password').value;
            const confirmPassword = document.getElementById('confirm_password').value;
            const username = document.getElementById('username').value;
            const submitBtn = document.getElementById('submitBtn');
            
            if (username.length >= 3 && password.length >= 6 && password === confirmPassword) {
                submitBtn.disabled = false;
                submitBtn.style.opacity = '1';
                submitBtn.style.cursor = 'pointer';
            } else {
                submitBtn.disabled = true;
                submitBtn.style.opacity = '0.6';
                submitBtn.style.cursor = 'not-allowed';
            }
        }

        // Check on username input too
        document.getElementById('username').addEventListener('input', updateSubmitButton);

        // Handle form submission with loading state
        document.getElementById('signupForm').addEventListener('submit', function(e) {
            const btn = this.querySelector('.signup-btn');
            const icon = btn.querySelector('i');
            const span = btn.querySelector('span');
            
            btn.disabled = true;
            icon.classList.remove('fa-user-plus');
            icon.classList.add('fa-spinner', 'fa-spin');
            span.textContent = 'Creating Account...';
        });

        // Auto-focus on username field
        document.addEventListener('DOMContentLoaded', function() {
            document.getElementById('username').focus();
        });
    </script>
</body>
</html>

## 🛠️ Spam Template (`templates/spam.html`)


In [ ]:
<!DOCTYPE html>
<html lang="en">

<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Spam Filter</title>
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
    <link rel="stylesheet" href="/style.css">

</head>

<body>
    <div class="container">
        <!-- Sidebar -->
        <aside class="sidebar">
            <div class="logo">
                <i class="fas fa-envelope-open-text"></i>
                <h2>Email Manager</h2>
            </div>
            <nav class="nav-menu">
                <a href="/" class="nav-item">
                    <i class="fas fa-inbox"></i>
                    <span>Inbox</span>
                </a>
                <a href="/analysis" class="nav-item">
                    <i class="fas fa-chart-line"></i>
                    <span>Analysis</span>
                </a>
                <a href="/reminders" class="nav-item">
                    <i class="fas fa-bell"></i>
                    <span>Reminders</span>
                </a>
                <a href="/spam" class="nav-item active">
                    <i class="fas fa-shield-alt"></i>
                    <span>Spam</span>
                </a>
            </nav>
            <div class="sidebar-footer">
                <button class="refresh-btn" onclick="generateTodoList()"
                    style="background: linear-gradient(135deg, #48bb78 0%, #38a169 100%); margin-bottom: 10px;">
                    <i class="fas fa-magic"></i>
                    <span>Generate To-Do List</span>
                </button>
                <button class="refresh-btn" onclick="openSendEmailModal()">
                    <i class="fas fa-paper-plane"></i>
                    <span>Send Email</span>
                </button>
                <button class="refresh-btn" onclick="openAIEmailModal()"
                    style="background: linear-gradient(135deg, #9f7aea 0%, #805ad5 100%); margin-top: 10px;">
                    <i class="fas fa-robot"></i>
                    <span>Generate AI Email</span>
                </button>
                <button class="refresh-btn" onclick="refreshEmails()" style="margin-top: 10px;">
                    <i class="fas fa-sync-alt"></i>
                    <span>Refresh Emails</span>
                </button>
                <button class="refresh-btn" onclick="window.location.href='/logout'"
                    style="background: linear-gradient(135deg, #f56565 0%, #e53e3e 100%); margin-top: 10px;">
                    <i class="fas fa-sign-out-alt"></i>
                    <span>Logout</span>
                </button>
                <p class="last-update">Last update: <span id="lastUpdate">--:--</span></p>
            </div>
        </aside>

        <!-- Main Content -->
        <main class="main-content">
            <header class="header">
                <div class="header-left">
                    <h1>Spam Detection</h1>
                    <p class="subtitle">Filtered suspicious emails</p>
                </div>
            </header>

            <!-- Spam Stats -->
            <div class="stats-grid">
                <div class="stat-card">
                    <div class="stat-icon red">
                        <i class="fas fa-exclamation-triangle"></i>
                    </div>
                    <div class="stat-info">
                        <h3>{{ spam_emails|length }}</h3>
                        <p>Spam Detected</p>
                    </div>
                </div>
                <div class="stat-card">
                    <div class="stat-icon green">
                        <i class="fas fa-shield-alt"></i>
                    </div>
                    <div class="stat-info">
                        <h3>{{ safe_emails }}</h3>
                        <p>Safe Emails</p>
                    </div>
                </div>
                <div class="stat-card">
                    <div class="stat-icon blue">
                        <i class="fas fa-percentage"></i>
                    </div>
                    <div class="stat-info">
                        <h3>{{ spam_percentage }}%</h3>
                        <p>Spam Rate</p>
                    </div>
                </div>
            </div>

            <!-- Spam List -->
            <div class="spam-container">
                <div class="section-header">
                    <h2 class="section-title">Detected Spam Emails</h2>
                    <div class="filter-buttons">
                        <button class="filter-btn active" onclick="filterSpam('all')">All Spam</button>
                        <button class="filter-btn" onclick="filterSpam('urgent')">Urgent</button>
                        <button class="filter-btn" onclick="filterSpam('promotions')">Promotions</button>
                    </div>
                </div>

                {% if spam_emails|length == 0 %}
                <div class="empty-state">
                    <i class="fas fa-check-circle"></i>
                    <h3>No Spam Detected</h3>
                    <p>All your emails appear to be legitimate.</p>
                </div>
                {% else %}
                <div class="spam-list">
                    {% for email in spam_emails %}
                    <div class="spam-card" data-category="{{ email.category|default('Other') }}">
                        <div class="spam-warning">
                            <i class="fas fa-exclamation-triangle"></i>
                            <span>SPAM DETECTED</span>
                        </div>

                        <div class="spam-header">
                            <div class="spam-sender">
                                <div class="avatar">
                                    <i class="fas fa-user-slash"></i>
                                </div>
                                <div class="sender-info">
                                    <h4>{{ email.from|default('Unknown Sender') }}</h4>
                                    <p class="email-time">{{ email.date|default('No Date') }}</p>
                                </div>
                            </div>
                            <div class="spam-badges">
                                <span class="badge badge-danger">
                                    <i class="fas fa-ban"></i> Spam
                                </span>
                                {% if email.category %}
                                <span class="badge badge-category">{{ email.category }}</span>
                                {% endif %}
                            </div>
                        </div>

                        <div class="spam-content">
                            <h3 class="email-subject">{{ email.subject|default('No Subject') }}</h3>
                            <p class="email-preview">{{ email.body[:200]|default('No content')|striptags }}...</p>
                        </div>

                        <div class="spam-footer">
                            <button class="btn-danger" onclick="permanentDelete('{{ email.id }}')">
                                <i class="fas fa-trash"></i> Delete Permanently
                            </button>
                            <button class="btn-secondary" onclick="markNotSpam('{{ email.id }}')">
                                <i class="fas fa-undo"></i> Not Spam
                            </button>
                            <button class="btn-view-spam" onclick="viewSpamDetails({{ loop.index0 }})">
                                <i class="fas fa-eye"></i> View Details
                            </button>
                        </div>
                    </div>
                    {% endfor %}
                </div>
                {% endif %}
            </div>

            <!-- Spam Protection Info -->
            <div class="protection-info">
                <div class="info-card">
                    <i class="fas fa-shield-alt"></i>
                    <h3>Advanced Protection</h3>
                    <p>Our AI-powered spam filter analyzes email content, sender reputation, and patterns to keep your
                        inbox clean.</p>
                </div>
                <div class="info-card">
                    <i class="fas fa-brain"></i>
                    <h3>Smart Learning</h3>
                    <p>The system continuously learns from your feedback to improve spam detection accuracy over time.
                    </p>
                </div>
                <div class="info-card">
                    <i class="fas fa-lock"></i>
                    <h3>Safe & Secure</h3>
                    <p>All suspicious emails are quarantined safely, protecting you from phishing and malicious content.
                    </p>
                </div>
            </div>
        </main>
    </div>

    <!-- Spam Detail Modal -->
    <div id="spamModal" class="modal">
        <div class="modal-content">
            <div class="modal-header">
                <h2 id="modalSubject">Spam Email Details</h2>
                <span class="close" onclick="closeModal()">&times;</span>
            </div>
            <div class="modal-body">
                <div class="warning-box">
                    <i class="fas fa-exclamation-triangle"></i>
                    <strong>Warning:</strong> This email has been identified as spam. Be cautious with any links or
                    attachments.
                </div>
                <div class="detail-row">
                    <strong>From:</strong>
                    <span id="modalFrom"></span>
                </div>
                <div class="detail-row">
                    <strong>Date:</strong>
                    <span id="modalDate"></span>
                </div>
                <div class="detail-row">
                    <strong>Category:</strong>
                    <span id="modalCategory"></span>
                </div>
                <div class="detail-row full-width">
                    <strong>Email Body:</strong>
                    <div class="email-body-full" id="modalBody"></div>
                </div>
                <!-- AI Actions -->
                <div class="modal-footer"
                    style="padding-top: 20px; border-top: 1px solid #e2e8f0; display: flex; gap: 10px; margin-top: 20px;">
                    <button class="btn-view" onclick="summarizeEmail()"
                        style="background: linear-gradient(135deg, #a0aec0 0%, #718096 100%);">
                        <i class="fas fa-compress-alt"></i> AI Summarize
                    </button>
                    <button class="btn-view" onclick="speakEmail(event)"
                        style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);">
                        <i class="fas fa-volume-up"></i> Read Aloud
                    </button>
                    <button class="btn-view" onclick="pauseSpeech()" id="pauseBtn"
                        style="display:none; background: #ecc94b; color: #744210;">
                        <i class="fas fa-pause"></i> Pause
                    </button>
                    <button class="btn-view" onclick="resumeSpeech()" id="resumeBtn"
                        style="display:none; background: #48bb78;">
                        <i class="fas fa-play"></i> Resume
                    </button>
                    <button class="btn-view" onclick="stopSpeech()" id="stopBtn"
                        style="display:none; background: #f56565;">
                        <i class="fas fa-stop"></i> Stop
                    </button>
                </div>
                <div id="summaryBox"
                    style="display: none; margin-top: 20px; padding: 15px; background: #f7fafc; border-radius: 8px; border-left: 4px solid #667eea;">
                    <strong>AI Summary:</strong>
                    <p id="summaryText" style="margin-top: 10px; font-style: italic;"></p>
                </div>
            </div>
        </div>

    </div>

    <!-- Modals (Copied from Index for consistency) -->
    <!-- Send Email Modal -->
    <div id="sendEmailModal" class="modal">
        <div class="modal-content">
            <span class="close-btn" onclick="closeModal('sendEmailModal')">&times;</span>
            <h2>Send New Email</h2>
            <form id="sendEmailForm" onsubmit="sendEmail(event)">
                <div class="form-group" style="margin-bottom: 15px;">
                    <label style="display: block; margin-bottom: 5px;">Recipient Email:</label>
                    <input type="email" id="emailRecipient" required
                        style="width: 100%; padding: 10px; border: 1px solid #ddd; border-radius: 5px;">
                </div>
                <div class="form-group" style="margin-bottom: 15px;">
                    <label style="display: block; margin-bottom: 5px;">Subject:</label>
                    <input type="text" id="emailSubject" required
                        style="width: 100%; padding: 10px; border: 1px solid #ddd; border-radius: 5px;">
                </div>
                <div class="form-group" style="margin-bottom: 15px;">
                    <label style="display: block; margin-bottom: 5px;">Message Body:</label>
                    <textarea id="emailBody" rows="8" required
                        style="width: 100%; padding: 10px; border: 1px solid #ddd; border-radius: 5px;"></textarea>
                </div>
                <button type="submit" class="btn-primary" style="width: 100%; padding: 12px; font-weight: 600;">
                    <i class="fas fa-paper-plane"></i> Send Email
                </button>
            </form>
        </div>
    </div>

    <!-- AI Email Generation Modal -->
    <div id="aiEmailModal" class="modal">
        <div class="modal-content">
            <span class="close-btn" onclick="closeModal('aiEmailModal')">&times;</span>
            <div style="display: flex; align-items: center; gap: 15px; margin-bottom: 20px;">
                <div
                    style="width: 50px; height: 50px; background: #9f7aea; border-radius: 12px; display: flex; align-items: center; justify-content: center; color: white; font-size: 24px;">
                    <i class="fas fa-robot"></i>
                </div>
                <div>
                    <h2 style="margin: 0;">AI Email Generator</h2>
                    <p style="margin: 5px 0 0; color: #718096; font-size: 14px;">Powered by Gemini 1.5</p>
                </div>
            </div>

            <form id="aiEmailForm" onsubmit="generateAIEmail(event)">
                <div class="form-group" style="margin-bottom: 15px;">
                    <label style="display: block; margin-bottom: 8px; font-weight: 600; color: #4a5568;">What kind of
                        email?</label>
                    <select id="emailType"
                        style="width: 100%; padding: 12px; border: 2px solid #edf2f7; border-radius: 10px; font-size: 15px;">
                        <option value="professional">Professional Follow-up</option>
                        <option value="formal">Formal Inquiry</option>
                        <option value="casual">Friendly Note</option>
                        <option value="meeting">Meeting Request</option>
                    </select>
                </div>
                <div class="form-group" style="margin-bottom: 20px;">
                    <label style="display: block; margin-bottom: 8px; font-weight: 600; color: #4a5568;">Main purpose or
                        key points:</label>
                    <textarea id="emailPurpose" rows="4"
                        placeholder="e.g. Follow up on the project proposal we discussed yesterday..."
                        style="width: 100%; padding: 12px; border: 2px solid #edf2f7; border-radius: 10px; font-size: 15px; resize: none;"></textarea>
                </div>
                <button type="submit" class="btn-primary"
                    style="width: 100%; padding: 14px; background: linear-gradient(135deg, #9f7aea 0%, #805ad5 100%); border: none; font-weight: 600; font-size: 16px;">
                    <i class="fas fa-sparkles"></i> Generate Draft
                </button>
            </form>

            <div id="aiEmailOutput"
                style="display: none; margin-top: 25px; border-top: 2px solid #edf2f7; padding-top: 20px;">
                <h3 style="font-size: 15px; color: #718096; margin-bottom: 10px;">AI Result:</h3>
                <div id="generatedEmail"
                    style="background: #f8fafc; padding: 20px; border-radius: 12px; border: 1px solid #e2e8f0; font-family: inherit; white-space: pre-wrap; margin-bottom: 15px; line-height: 1.6; color: #2d3748;">
                </div>
                <button onclick="copyToClipboard()" class="btn-view" style="width: 100%; justify-content: center;">
                    <i class="fas fa-copy"></i> Copy to Clipboard
                </button>
            </div>
        </div>
    </div>

    <script>
        const spamEmails = {{ spam_emails| tojson }};

        function filterSpam(type) {
            const cards = document.querySelectorAll('.spam-card');
            const buttons = document.querySelectorAll('.filter-btn');

            buttons.forEach(btn => btn.classList.remove('active'));
            event.target.classList.add('active');

            cards.forEach(card => {
                if (type === 'all') {
                    card.style.display = 'block';
                } else if (type === 'urgent') {
                    card.style.display = card.querySelector('.badge-warning') ? 'block' : 'none';
                } else if (type === 'promotions') {
                    const category = card.getAttribute('data-category');
                    card.style.display = category === 'Promotions' ? 'block' : 'none';
                }
            });
        }

        function viewSpamDetails(index) {
            const email = spamEmails[index];
            document.getElementById('modalSubject').textContent = email.subject || 'No Subject';
            document.getElementById('modalFrom').textContent = email.from || 'Unknown';
            document.getElementById('modalDate').textContent = email.date || 'No Date';
            document.getElementById('modalCategory').textContent = email.category || 'Other';
            document.getElementById('modalBody').textContent = email.body || 'No content';
            document.getElementById('spamModal').style.display = 'block';
        }

        function closeModal() {
            document.getElementById('spamModal').style.display = 'none';
            document.getElementById('summaryBox').style.display = 'none';
            window.speechSynthesis.cancel();
        }

        async function summarizeEmail() {
            const body = document.getElementById('modalBody').textContent;
            const summaryBox = document.getElementById('summaryBox');
            const summaryText = document.getElementById('summaryText');

            summaryText.textContent = "Summarizing...";
            summaryBox.style.display = 'block';

            try {
                const response = await fetch('/api/summarize', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ body })
                });
                const result = await response.json();
                if (result.success) {
                    summaryText.textContent = result.summary;
                } else {
                    summaryText.textContent = "Error: " + result.message;
                }
            } catch (error) {
                summaryText.textContent = "Failed to fetch summary.";
            }
        }

        async function speakEmail(event) {
            const body = document.getElementById('modalBody').textContent;
            const speakBtn = event.currentTarget;
            const originalHTML = speakBtn.innerHTML;

            // Show loading state
            speakBtn.innerHTML = '<i class="fas fa-spinner fa-spin"></i> Processing...';
            speakBtn.disabled = true;

            try {
                const response = await fetch('/api/clean_text', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ body })
                });
                const result = await response.json();

                if (result.success) {
                    window.speechSynthesis.cancel();
                    const utterance = new SpeechSynthesisUtterance(result.cleaned_text);

                    utterance.onstart = () => {
                        document.getElementById('pauseBtn').style.display = 'flex';
                        document.getElementById('stopBtn').style.display = 'flex';
                        document.getElementById('resumeBtn').style.display = 'none';
                    };

                    utterance.onend = () => {
                        document.getElementById('pauseBtn').style.display = 'none';
                        document.getElementById('stopBtn').style.display = 'none';
                        document.getElementById('resumeBtn').style.display = 'none';
                    };

                    window.speechSynthesis.speak(utterance);
                } else {
                    alert('Error cleaning text: ' + result.message);
                }
            } catch (error) {
                console.error(error);
                alert('Failed to process text for reading.');
            } finally {
                speakBtn.innerHTML = originalHTML;
                speakBtn.disabled = false;
            }
        }

        function pauseSpeech() {
            window.speechSynthesis.pause();
            document.getElementById('pauseBtn').style.display = 'none';
            document.getElementById('resumeBtn').style.display = 'flex';
        }

        function resumeSpeech() {
            window.speechSynthesis.resume();
            document.getElementById('pauseBtn').style.display = 'flex';
            document.getElementById('resumeBtn').style.display = 'none';
        }

        function stopSpeech() {
            window.speechSynthesis.cancel();
            document.getElementById('pauseBtn').style.display = 'none';
            document.getElementById('stopBtn').style.display = 'none';
            document.getElementById('resumeBtn').style.display = 'none';
        }

        function permanentDelete(emailId) {
            if (confirm('Are you sure you want to permanently delete this spam email?')) {
                alert('Email deleted successfully!');
                window.location.reload();
            }
        }

        function markNotSpam(emailId) {
            if (confirm('Mark this email as not spam?')) {
                alert('Email marked as safe!');
                window.location.reload();
            }
        }

        function openSendEmailModal() {
            document.getElementById('sendEmailModal').style.display = 'block';
        }

        function openAIEmailModal() {
            document.getElementById('aiEmailModal').style.display = 'block';
            document.getElementById('aiEmailOutput').style.display = 'none';
        }

        async function sendEmail(event) {
            event.preventDefault();
            const recipient = document.getElementById('emailRecipient').value;
            const subject = document.getElementById('emailSubject').value;
            const body = document.getElementById('emailBody').value;
            try {
                const response = await fetch('/api/send_email', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ recipient, subject, body })
                });
                const result = await response.json();
                if (result.success) {
                    alert('Email logged successfully!');
                    closeModal('sendEmailModal');
                    document.getElementById('sendEmailForm').reset();
                } else {
                    alert('Error: ' + result.message);
                }
            } catch (error) {
                console.error('Error:', error);
                alert('Failed to send email');
            }
        }

        async function generateAIEmail(event) {
            event.preventDefault();
            const emailType = document.getElementById('emailType').value;
            const purpose = document.getElementById('emailPurpose').value;
            try {
                const response = await fetch('/api/generate_ai_email', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ email_type: emailType, purpose })
                });
                const result = await response.json();
                if (result.success) {
                    document.getElementById('generatedEmail').textContent = result.email;
                    document.getElementById('aiEmailOutput').style.display = 'block';
                } else {
                    alert('Error: ' + result.message);
                }
            } catch (error) {
                console.error('Error:', error);
                alert('Failed to generate email');
            }
        }

        async function generateTodoList() {
            if (!confirm('Generate AI-enhanced to-do list from today\'s reminders?')) return;
            window.location.href = '/reminders?generate=true';
        }

        function copyToClipboard() {
            const text = document.getElementById('generatedEmail').textContent;
            navigator.clipboard.writeText(text).then(() => {
                alert('Email copied to clipboard!');
            });
        }

        // Refresh emails
        async function refreshEmails() {
            const btn = document.querySelector('.refresh-btn');
            btn.classList.add('spinning');
            try {
                const response = await fetch('/api/refresh', { method: 'POST' });
                if (response.ok) {
                    window.location.reload();
                }
            } catch (error) {
                console.error('Error refreshing:', error);
            }
            setTimeout(() => btn.classList.remove('spinning'), 1000);
        }

        // Update last update time
        function updateLastUpdateTime() {
            fetch('/api/stats')
                .then(response => response.json())
                .then(data => {
                    const el = document.getElementById('lastUpdate');
                    if (el) el.textContent = data.last_update || '--:--';
                })
                .catch(error => console.error('Error fetching stats:', error));
        }

        // Auto-refresh stats every 30 seconds
        setInterval(updateLastUpdateTime, 30000);
        updateLastUpdateTime();

        window.onclick = function (event) {
            if (event.target.classList.contains('modal')) {
                event.target.style.display = 'none';
            }
        }
    </script>
</body>

</html>

## 🛠️ Analysis Template (`templates/analysis.html`)


In [ ]:
<!DOCTYPE html>
<html lang="en">

<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Email Analysis</title>
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <link rel="stylesheet" href="/style.css">
</head>

<body>
    <div class="container">
        <!-- Sidebar -->
        <aside class="sidebar">
            <div class="logo">
                <i class="fas fa-envelope-open-text"></i>
                <h2>Email Manager</h2>
            </div>
            <nav class="nav-menu">
                <a href="/" class="nav-item">
                    <i class="fas fa-inbox"></i>
                    <span>Inbox</span>
                </a>
                <a href="/analysis" class="nav-item active">
                    <i class="fas fa-chart-line"></i>
                    <span>Analysis</span>
                </a>
                <a href="/reminders" class="nav-item">
                    <i class="fas fa-bell"></i>
                    <span>Reminders</span>
                </a>
                <a href="/spam" class="nav-item">
                    <i class="fas fa-shield-alt"></i>
                    <span>Spam</span>
                </a>
            </nav>
            <div class="sidebar-footer">
                <button class="refresh-btn" onclick="generateTodoList()"
                    style="background: linear-gradient(135deg, #48bb78 0%, #38a169 100%); margin-bottom: 10px;">
                    <i class="fas fa-magic"></i>
                    <span>Generate To-Do List</span>
                </button>
                <button class="refresh-btn" onclick="openSendEmailModal()">
                    <i class="fas fa-paper-plane"></i>
                    <span>Send Email</span>
                </button>
                <button class="refresh-btn" onclick="openAIEmailModal()"
                    style="background: linear-gradient(135deg, #9f7aea 0%, #805ad5 100%); margin-top: 10px;">
                    <i class="fas fa-robot"></i>
                    <span>Generate AI Email</span>
                </button>
                <button class="refresh-btn" onclick="refreshEmails()" style="margin-top: 10px;">
                    <i class="fas fa-sync-alt"></i>
                    <span>Refresh Emails</span>
                </button>
                <button class="refresh-btn" onclick="window.location.href='/logout'"
                    style="background: linear-gradient(135deg, #f56565 0%, #e53e3e 100%); margin-top: 10px;">
                    <i class="fas fa-sign-out-alt"></i>
                    <span>Logout</span>
                </button>
                <p class="last-update">Last update: <span id="lastUpdate">--:--</span></p>
            </div>
        </aside>

        <!-- Main Content -->
        <main class="main-content">
            <header class="header">
                <div class="header-left">
                    <h1>Email Analysis</h1>
                    <p class="subtitle">Insights and trends from your emails</p>
                </div>
            </header>

            <!-- Analysis Stats -->
            <div class="stats-grid">
                <div class="stat-card">
                    <div class="stat-icon blue">
                        <i class="fas fa-smile"></i>
                    </div>
                    <div class="stat-info">
                        <h3 id="positiveCount">{{ positive_count }}</h3>
                        <p>Positive Emails</p>
                    </div>
                </div>
                <div class="stat-card">
                    <div class="stat-icon red">
                        <i class="fas fa-frown"></i>
                    </div>
                    <div class="stat-info">
                        <h3 id="negativeCount">{{ negative_count }}</h3>
                        <p>Negative Emails</p>
                    </div>
                </div>
                <div class="stat-card">
                    <div class="stat-icon orange">
                        <i class="fas fa-meh"></i>
                    </div>
                    <div class="stat-info">
                        <h3 id="neutralCount">{{ neutral_count }}</h3>
                        <p>Neutral Emails</p>
                    </div>
                </div>
                <div class="stat-card">
                    <div class="stat-icon green">
                        <i class="fas fa-tags"></i>
                    </div>
                    <div class="stat-info">
                        <h3>{{ categories|length }}</h3>
                        <p>Categories</p>
                    </div>
                </div>
            </div>

            <!-- Charts Section -->
            <div class="charts-container">
                <div class="chart-card">
                    <h3>Email Categories Distribution</h3>
                    <canvas id="categoryChart"></canvas>
                </div>
                <div class="chart-card">
                    <h3>Sentiment Analysis</h3>
                    <canvas id="sentimentChart"></canvas>
                </div>
            </div>

            <div class="charts-container">
                <div class="chart-card">
                    <h3>Urgency Levels</h3>
                    <canvas id="urgencyChart"></canvas>
                </div>
                <div class="chart-card">
                    <h3>Email Activity Timeline</h3>
                    <canvas id="timelineChart"></canvas>
                </div>
            </div>

            <!-- Category Breakdown -->
            <div class="category-breakdown">
                <h2>Category Breakdown</h2>
                <div class="category-list">
                    {% for category, count in categories.items() %}
                    <div class="category-item">
                        <div class="category-name">
                            <i class="fas fa-folder"></i>
                            <span>{{ category }}</span>
                        </div>
                        <div class="category-count">{{ count }} emails</div>
                    </div>
                    {% endfor %}
                </div>
            </div>
        </main>
    </div>

    <!-- Modals -->
    <!-- Send Email Modal -->
    <div id="sendEmailModal" class="modal"
        style="display: none; position: fixed; z-index: 1000; left: 0; top: 0; width: 100%; height: 100%; background: rgba(0,0,0,0.5);">
        <div class="modal-content"
            style="background: white; margin: 10% auto; padding: 30px; border-radius: 15px; width: 50%; max-width: 600px; position: relative;">
            <span class="close-btn" onclick="closeModal('sendEmailModal')"
                style="position: absolute; right: 20px; top: 20px; font-size: 24px; cursor: pointer;">&times;</span>
            <h2>Send New Email</h2>
            <form id="sendEmailForm" onsubmit="sendEmail(event)">
                <div class="form-group" style="margin-bottom: 15px;">
                    <label style="display: block; margin-bottom: 5px;">Recipient Email:</label>
                    <input type="email" id="emailRecipient" required
                        style="width: 100%; padding: 10px; border: 1px solid #ddd; border-radius: 5px;">
                </div>
                <div class="form-group" style="margin-bottom: 15px;">
                    <label style="display: block; margin-bottom: 5px;">Subject:</label>
                    <input type="text" id="emailSubject" required
                        style="width: 100%; padding: 10px; border: 1px solid #ddd; border-radius: 5px;">
                </div>
                <div class="form-group" style="margin-bottom: 15px;">
                    <label style="display: block; margin-bottom: 5px;">Message Body:</label>
                    <textarea id="emailBody" rows="8" required
                        style="width: 100%; padding: 10px; border: 1px solid #ddd; border-radius: 5px;"></textarea>
                </div>
                <button type="submit" class="btn-primary"
                    style="width: 100%; padding: 12px; background: #667eea; color: white; border: none; border-radius: 8px; font-weight: 600; cursor: pointer;">
                    <i class="fas fa-paper-plane"></i> Send Email
                </button>
            </form>
        </div>
    </div>

    <!-- AI Email Generation Modal -->
    <div id="aiEmailModal" class="modal"
        style="display: none; position: fixed; z-index: 1000; left: 0; top: 0; width: 100%; height: 100%; background: rgba(0,0,0,0.5);">
        <div class="modal-content"
            style="background: white; margin: 10% auto; padding: 30px; border-radius: 15px; width: 50%; max-width: 600px; position: relative;">
            <span class="close-btn" onclick="closeModal('aiEmailModal')"
                style="position: absolute; right: 20px; top: 20px; font-size: 24px; cursor: pointer;">&times;</span>
            <div style="display: flex; align-items: center; gap: 15px; margin-bottom: 20px;">
                <div
                    style="width: 50px; height: 50px; background: #9f7aea; border-radius: 12px; display: flex; align-items: center; justify-content: center; color: white; font-size: 24px;">
                    <i class="fas fa-robot"></i>
                </div>
                <div>
                    <h2 style="margin: 0;">AI Email Generator</h2>
                    <p style="margin: 5px 0 0; color: #718096; font-size: 14px;">Powered by Gemini</p>
                </div>
            </div>

            <form id="aiEmailForm" onsubmit="generateAIEmail(event)">
                <div class="form-group" style="margin-bottom: 15px;">
                    <label style="display: block; margin-bottom: 8px; font-weight: 600; color: #4a5568;">What kind of
                        email?</label>
                    <select id="emailType"
                        style="width: 100%; padding: 12px; border: 2px solid #edf2f7; border-radius: 10px; font-size: 15px;">
                        <option value="professional">Professional Follow-up</option>
                        <option value="formal">Formal Inquiry</option>
                        <option value="casual">Friendly Note</option>
                        <option value="meeting">Meeting Request</option>
                    </select>
                </div>
                <div class="form-group" style="margin-bottom: 20px;">
                    <label style="display: block; margin-bottom: 8px; font-weight: 600; color: #4a5568;">Main purpose or
                        key points:</label>
                    <textarea id="emailPurpose" rows="4"
                        placeholder="e.g. Follow up on the project proposal we discussed yesterday..."
                        style="width: 100%; padding: 12px; border: 2px solid #edf2f7; border-radius: 10px; font-size: 15px; resize: none;"></textarea>
                </div>
                <button type="submit" class="btn-primary"
                    style="width: 100%; padding: 14px; background: linear-gradient(135deg, #9f7aea 0%, #805ad5 100%); border: none; color: white; border-radius: 8px; font-weight: 600; font-size: 16px; cursor: pointer;">
                    <i class="fas fa-sparkles"></i> Generate Draft
                </button>
            </form>

            <div id="aiEmailOutput"
                style="display: none; margin-top: 25px; border-top: 2px solid #edf2f7; padding-top: 20px;">
                <h3 style="font-size: 15px; color: #718096; margin-bottom: 10px;">AI Result:</h3>
                <div id="generatedEmail"
                    style="background: #f8fafc; padding: 20px; border-radius: 12px; border: 1px solid #e2e8f0; font-family: inherit; white-space: pre-wrap; margin-bottom: 15px; line-height: 1.6; color: #2d3748;">
                </div>
                <button onclick="copyToClipboard()"
                    style="width: 100%; padding: 10px; background: #e2e8f0; border: none; border-radius: 8px; cursor: pointer; display: flex; align-items: center; justify-content: center; gap: 8px;">
                    <i class="fas fa-copy"></i> Copy to Clipboard
                </button>
            </div>
        </div>
    </div>

    <script>
        // Category Chart
        const categoryCtx = document.getElementById('categoryChart').getContext('2d');
        new Chart(categoryCtx, {
            type: 'doughnut',
            data: {
                labels: {{ category_labels| tojson | safe }},
            datasets: [{
                data: {{ category_data| tojson | safe }},
            backgroundColor: [
                '#667eea', '#48bb78', '#ed8936', '#f56565',
                '#4299e1', '#9f7aea', '#38b2ac', '#ed64a6'
            ]
                }]
            },
            options: {
            responsive: true,
            maintainAspectRatio: false,
            plugins: {
                legend: {
                    position: 'bottom'
                }
            }
        }
        });

        // Sentiment Chart
        const sentimentCtx = document.getElementById('sentimentChart').getContext('2d');
        new Chart(sentimentCtx, {
            type: 'bar',
            data: {
                labels: ['Positive', 'Neutral', 'Negative'],
                datasets: [{
                    label: 'Number of Emails',
                    data: [{{ positive_count }}, {{ neutral_count }}, {{ negative_count }}],
            backgroundColor: ['#48bb78', '#ed8936', '#f56565']
                }]
            },
            options: {
            responsive: true,
            maintainAspectRatio: false,
            scales: {
                y: {
                    beginAtZero: true
                }
            }
        }
        });

        // Urgency Chart
        const urgencyCtx = document.getElementById('urgencyChart').getContext('2d');
        new Chart(urgencyCtx, {
            type: 'pie',
            data: {
                labels: {{ urgency_labels| tojson | safe }},
            datasets: [{
                data: {{ urgency_data| tojson | safe }},
            backgroundColor: ['#f56565', '#ed8936', '#48bb78']
                }]
            },
            options: {
            responsive: true,
            maintainAspectRatio: false,
            plugins: {
                legend: {
                    position: 'bottom'
                }
            }
        }
        });

        // Timeline Chart
        const timelineCtx = document.getElementById('timelineChart').getContext('2d');
        new Chart(timelineCtx, {
            type: 'line',
            data: {
                labels: {{ timeline_labels| tojson }},
            datasets: [{
                label: 'Emails Received',
                data: {{ timeline_data| tojson }},
            borderColor: '#667eea',
            backgroundColor: 'rgba(102, 126, 234, 0.1)',
            tension: 0.4,
            fill: true
                }]
            },
            options: {
            responsive: true,
            maintainAspectRatio: false,
            scales: {
                y: {
                    beginAtZero: true
                }
            }
        }
        });

        // Common Navigation Handlers
        function closeModal(modalId) {
            document.getElementById(modalId).style.display = 'none';
        }

        function openSendEmailModal() {
            document.getElementById('sendEmailModal').style.display = 'block';
        }

        function openAIEmailModal() {
            document.getElementById('aiEmailModal').style.display = 'block';
            document.getElementById('aiEmailOutput').style.display = 'none';
        }

        async function sendEmail(event) {
            event.preventDefault();
            const recipient = document.getElementById('emailRecipient').value;
            const subject = document.getElementById('emailSubject').value;
            const body = document.getElementById('emailBody').value;
            try {
                const response = await fetch('/api/send_email', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ recipient, subject, body })
                });
                const result = await response.json();
                if (result.success) {
                    alert('Email logged successfully!');
                    closeModal('sendEmailModal');
                    document.getElementById('sendEmailForm').reset();
                } else {
                    alert('Error: ' + result.message);
                }
            } catch (error) {
                console.error('Error:', error);
                alert('Failed to send email');
            }
        }

        async function generateAIEmail(event) {
            event.preventDefault();
            const emailType = document.getElementById('emailType').value;
            const purpose = document.getElementById('emailPurpose').value;
            try {
                const response = await fetch('/api/generate_ai_email', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ email_type: emailType, purpose })
                });
                const result = await response.json();
                if (result.success) {
                    document.getElementById('generatedEmail').textContent = result.email;
                    document.getElementById('aiEmailOutput').style.display = 'block';
                } else {
                    alert('Error: ' + result.message);
                }
            } catch (error) {
                console.error('Error:', error);
                alert('Failed to generate email');
            }
        }

        async function generateTodoList() {
            if (!confirm('Generate AI-enhanced to-do list from today\'s reminders?')) return;
            window.location.href = '/reminders?generate=true';
        }

        function copyToClipboard() {
            const text = document.getElementById('generatedEmail').textContent;
            navigator.clipboard.writeText(text).then(() => {
                alert('Email copied to clipboard!');
            });
        }

        async function refreshEmails() {
            const btn = document.querySelector('.refresh-btn');
            btn.classList.add('spinning');
            try {
                const response = await fetch('/api/refresh', { method: 'POST' });
                if (response.ok) {
                    window.location.reload();
                }
            } catch (error) {
                console.error('Error refreshing:', error);
            }
            setTimeout(() => btn.classList.remove('spinning'), 1000);
        }

        function updateLastUpdateTime() {
            fetch('/api/stats')
                .then(response => response.json())
                .then(data => {
                    const el = document.getElementById('lastUpdate');
                    if (el) el.textContent = data.last_update || '--:--';
                })
                .catch(error => console.error('Error fetching stats:', error));
        }

        setInterval(updateLastUpdateTime, 30000);
        updateLastUpdateTime();

        window.onclick = function (event) {
            if (event.target.classList.contains('modal')) {
                event.target.style.display = 'none';
            }
        }
    </script>
</body>

</html>